# Detección de fraude en pagos · El modelo (TensorFlow)

**ITY1102 · Arquitectura de Sistemas de IA** — Caso 3: *Kipu Pagos SpA* · decisión **aprobar / revisar / bloquear** sobre cada transacción, explicable y auditable.

> Este notebook sigue la estructura del notebook base (*Del notebook al microservicio · Parte 1*), pero el caso de fraude tiene trampas que el caso de camiones no tenía. Cada una cambia una decisión de diseño:

| Trampa del caso (sección 8 y 9) | Qué haría un notebook ingenuo | Qué hacemos aquí |
|---|---|---|
| **R2 · Fuga de información**: `revisado_por_analista` y `motivo_bloqueo` se llenan *después* del veredicto | Entrenarlas y obtener ~100% de exactitud | Se demuestra la fuga y se excluyen del modelo |
| **R3 · Muestra estratificada**: 0,5% de fraude en el archivo vs 0,012% en producción | Calibrar umbrales y costos sobre el 0,5% | Se re-ponderan las métricas a la proporción real y se corrige la probabilidad (*prior shift*) |
| **Etiqueta vacía ≠ legítima** (R6) | `fillna(0)` sin pensar | Se separa «confirmada legítima» de «no revisada», se mide el efecto de tratarlas distinto |
| **Zonas horarias mezcladas** | Parsear y seguir | Se **verifica con los datos** qué zona tienen las fechas sin zona (UTC-4, hora de Chile) |
| **Duplicados en segundos**: reintentos legítimos vs duplicados del sistema | `drop_duplicates()` | Dos reglas distintas: el duplicado del sistema se borra, el reintento se conserva |
| **RT8 · Explicabilidad** + **RF2** (razones ordenadas por peso) | Una red opaca + SHAP aproximado después | Una **red neuronal aditiva** (una subred por factor): el aporte de cada factor es **exacto**, no una aproximación |
| **RF1 · tres decisiones**, **RF3 · umbrales sin desplegar**, **DA5 · los fija riesgo** | Un solo umbral 0,5 en el código | Dos umbrales en `umbrales.json`, separados del modelo, con la frontera costo–riesgo para que **riesgo** decida |
| **R4 · sin turno de madrugada** (49% del fraude ocurre 00–06) | Asumir que «revisar» siempre sirve | «Revisar» de madrugada llega tarde (ya se liquidó): la política de noche solo tiene *aprobar* o *bloquear* |

```
 [ notebook ]  ──►  preprocesador.joblib + modelo.onnx + metadata.json + umbrales.json  ──►  [ FastAPI + onnxruntime ]  ──► ...
   TensorFlow          (artefacto versionado, sin TensorFlow adentro)                          (parte 2)
```

### Antes de empezar: dos aclaraciones sobre las cifras del caso

1. **Las cifras de la sección 2 no cuadran entre sí.** Con 200.000 transacciones/día y 0,012% de fraude hay ~24 fraudes diarios (y USD 500.000/año ≈ 24 × 365 × USD 57, que sí cuadra). Pero «1.150 bloqueos/día, 40% legítimos» implicaría ~690 fraudes bloqueados al día. Probablemente los bloqueos del motor actual incluyen causas que no son fraude. **Usamos la prevalencia de 0,012% de la sección 8**, que es la que sostiene el resto de los números.
2. **Los costos que el caso no da son supuestos explícitos** (están marcados abajo con `SUPUESTO`): costo de una revisión manual, efectividad del analista y tipo de cambio. Todos son parámetros; si cambian, se re-ejecuta.

## 0 · Preparación

In [ ]:
# TensorFlow ya viene instalado en Colab. Agregamos ONNX (formato), onnxruntime (para verificar como lo hará la API),
# tf2onnx (lo usa Keras para exportar) y la MISMA versión de scikit-learn que la imagen.
!pip install -q scikit-learn==1.9.1 onnx onnxruntime==1.30.0 tf2onnx

In [ ]:
import hashlib
import json
import re
import sys
import time
import unicodedata
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

FRAMEWORK = "tensorflow"
SEMILLA = 42
EN_COLAB = "google.colab" in sys.modules
CARPETA = Path("/content") if EN_COLAB else Path(".")
RUTA_MODELO = CARPETA / "model"          # aquí se exporta preprocesador.joblib + modelo.onnx + metadata.json + umbrales.json

# ---------- Cifras del caso (sección 2 y 8) ----------
TX_DIA = 200_000                         # transacciones diarias
PI_PROD = 0.00012                        # 0,012% de fraude en producción (sección 8)
FRAUDES_DIA = TX_DIA * PI_PROD           # ≈ 24 fraudes al día
LEGITIMAS_DIA = TX_DIA - FRAUDES_DIA
COSTO_FP_BLOQUEO = 1_162_600 / (460 * 365)   # USD por cliente bueno bloqueado (tabla de la sección 2) ≈ 6,92
CAPACIDAD_REVISION_DIA = 4_800           # lo que hoy revisan los 6 analistas (2,4% del tráfico)
TURNO = (8, 22)                          # analistas de 08:00 a 22:00, hora de Chile (RO2)
FP_OBJETIVO = 0.15                       # meta: falsos positivos sobre bloqueos < 15%
FRAUDE_NO_DET_OBJETIVO = 350_000         # meta: fraude no detectado < USD 350.000/año

# ---------- SUPUESTOS (el caso no los entrega) ----------
TASA_CLP_USD = 950                       # SUPUESTO: USD 26 ≈ $24.700 CLP (sección 2)
COSTO_REVISION = 0.10                    # SUPUESTO: USD por revisión manual (6 analistas ≈ USD 12.000/mes sobre 144.000 revisiones/mes)
EFECTIVIDAD_REVISION = 0.90              # SUPUESTO: fracción de los fraudes revisados dentro del turno que el analista confirma
PESO_FRAUDE = 20.0                       # peso de la clase fraude al entrenar (se recalibra después, ver sección 8)
PESO_NO_REVISADAS = 1.0                  # peso de las filas sin etiqueta tratadas como legítimas (se mide su efecto en 7.2)

print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
print(f"Fraudes por día en producción: {FRAUDES_DIA:.0f} | costo de bloquear a un cliente bueno: USD {COSTO_FP_BLOQUEO:.2f}")

## 1 · Cargar los datos

Se usan los 4 CSV de la muestra (`usuarios`, `transacciones`, `dispositivos`, `lista_negra`). En Colab, si no están en `/content`, la celda pide subirlos.

In [ ]:
ARCHIVOS = ["usuarios.csv", "transacciones.csv", "lista_negra.csv", "dispositivos.csv"]

def ubicar_datos() -> Path:
    for c in [CARPETA, Path("."), Path("data"), Path("/mnt/user-data/uploads")]:
        if all((c / a).exists() for a in ARCHIVOS):
            return c
    if EN_COLAB:
        from google.colab import files
        print("Sube los 4 CSV:", ARCHIVOS)
        files.upload()
        return Path(".")
    raise FileNotFoundError(f"No encuentro {ARCHIVOS}; deja los CSV junto al notebook o en ./data")

RUTA_DATOS = ubicar_datos()
usuarios_raw = pd.read_csv(RUTA_DATOS / "usuarios.csv")
transacciones_raw = pd.read_csv(RUTA_DATOS / "transacciones.csv")
lista_negra_raw = pd.read_csv(RUTA_DATOS / "lista_negra.csv")
dispositivos_raw = pd.read_csv(RUTA_DATOS / "dispositivos.csv")

# Huella de los datos de entrada: queda en metadata.json (RF7: "qué datos")
HASH_DATOS = {a: hashlib.sha256((RUTA_DATOS / a).read_bytes()).hexdigest()[:16] for a in ARCHIVOS}
print("Datos en:", RUTA_DATOS.resolve())
pd.DataFrame({"filas": {"usuarios": len(usuarios_raw), "transacciones": len(transacciones_raw),
                        "lista_negra": len(lista_negra_raw), "dispositivos": len(dispositivos_raw)}})

## 2 · Perfilado: lo que condiciona el modelo antes de tocar un dato

### 2.1 La etiqueta: vacío no significa «legítima»

In [ ]:
def clave(x):
    # minúsculas, sin tildes, sin espacios sobrantes: 'Perú ' -> 'peru'
    if pd.isna(x):
        return x
    s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

BOOL = {"1": 1, "0": 0, "1.0": 1, "0.0": 0, "si": 1, "no": 0, "true": 1, "false": 0}

t = transacciones_raw.copy()
t["fraude"] = t["es_fraude"].map(clave).map(BOOL)
t["revisada"] = t["revisado_por_analista"].map(clave).map(BOOL)

print("es_fraude por estado de revisión (filas de la muestra, sin quitar duplicados):")
print(pd.crosstab(t["revisada"].fillna(-1).map({1: "revisada", -1: "NO revisada"}),
                  t["fraude"].fillna(-1).map({1: "fraude", 0: "legítima", -1: "sin etiqueta"})))
print(f"\nFilas con etiqueta: {t['fraude'].notna().sum():,} de {len(t):,} ({t['fraude'].notna().mean():.1%}).")
print(f"Tasa de fraude sobre TODA la muestra: {t['fraude'].fillna(0).mean():.2%} (en producción: {PI_PROD:.3%})")

**Cómo leerlo.** La etiqueta solo existe donde un analista revisó (4,4% de las filas). El resto —96%— **no es «legítima confirmada»: es «nadie miró»**. Tratarlas como legítimas es una decisión con consecuencias (en producción el 0,012% sí es fraude, y parte de ese fraude está sin confirmar), así que la medimos en la sección 7.2 en vez de esconderla en un `fillna(0)`.

### 2.2 Las dos columnas que filtran el resultado
`revisado_por_analista` y `motivo_bloqueo` se llenan **después** de que el analista decide. En producción, al momento de autorizar, no existen.

In [ ]:
lab = t[t["fraude"].notna()]
print("motivo_bloqueo por veredicto (solo filas etiquetadas):")
print(pd.crosstab(lab["motivo_bloqueo"].notna().map({True: "motivo_bloqueo lleno", False: "motivo_bloqueo vacío"}),
                  lab["fraude"].map({1.0: "fraude", 0.0: "legítima"})))
auc_fuga = roc_auc_score(lab["fraude"], lab["motivo_bloqueo"].notna())
print(f"\nROC-AUC de predecir fraude con SOLO «¿motivo_bloqueo está lleno?»: {auc_fuga:.3f}")
print("→ Un modelo que vea esa columna 'aprende' el 100%... y es inútil: en producción siempre llega vacía.")

**Decisión:** `revisado_por_analista`, `motivo_bloqueo` y `es_fraude` (es la etiqueta, no una variable) **no entran** al modelo. Se conservan en el CSV solo para construir la etiqueta y, en el futuro, para el ciclo de reentrenamiento (RF6).

### 2.3 Otros hallazgos (los detalla la limpieza de la sección 3)
- `timestamp` viene en **4 formatos**; unos con zona (`Z`, `-04:00`) y otros sin ella.
- `monto` mezcla `$`, coma decimal, punto de miles y negativos; `moneda` está vacía en ~4% de las filas.
- Coordenadas `(0, 0)`, IPs inválidas, categorías escritas de muchas formas (ver notebook de limpieza).
- Transacciones repetidas en segundos.

## 3 · Limpieza con reglas escritas

Cada problema tiene **una regla**, y cada regla decide si la fila se **corrige**, se **descarta** o se **marca** (igual que en el notebook base).

| Regla | Problema | Decisión | Por qué |
|---|---|---|---|
| R1 | Misma `transaccion_id` repetida (copia exacta) | Descartar | Es la misma fila dos veces |
| R2 | Categorías mal escritas (`CL`, `chile `, `Chile`, `movil`…) | Corregir | Mismo valor, distinta escritura |
| R3 | Fechas sin zona horaria | Corregir a UTC | **Se verifica** que son hora de Chile (UTC-4); si se leyeran como UTC, la «hora del día» quedaría corrida 4 horas |
| R4 | `monto` en formatos mezclados; `moneda` vacía | Corregir | En CLP el `.` es separador de miles; en USD es decimal. Con eso se infiere la moneda |
| R5 | Coordenadas `(0, 0)` | Marcar como nulo | Es el Golfo de Guinea, un valor centinela |
| R6 | IP vacía o inválida | Marcar como nulo | `-`, octetos > 255, IPs truncadas |
| R7a | **Duplicado del sistema**: mismo pago aprobado dos veces en segundos | Descartar el segundo | Inflaría el entrenamiento y duplicaría el monto en riesgo |
| R7b | **Reintento del cliente**: mismo pago rechazado/pendiente, repetido en segundos | **Conservar** y marcar | Es un intento real; borrarlo esconde justo el patrón de velocidad |
| R8 | `es_fraude` vacío | **No** es «legítima» | Se guarda como «sin etiqueta»; el peso con que entra a entrenar es una decisión explícita |
| R9 | Usuario con `fecha_registro` futura, `nivel_kyc = -1`, `edad_cuenta_dias` negativa | Marcar como nulo | Valores imposibles; el nulo lo imputa el preprocesador |

In [ ]:
PAIS = {"cl": "CL", "chile": "CL", "ar": "AR", "argentina": "AR", "pe": "PE", "peru": "PE",
        "br": "BR", "brazil": "BR", "brasil": "BR", "co": "CO", "colombia": "CO",
        "ve": "VE", "venezuela": "VE", "us": "US", "usa": "US", "eeuu": "US", "estados unidos": "US",
        "ru": "RU", "russia": "RU", "rusia": "RU", "ng": "NG", "nigeria": "NG"}
SEGMENTO = {"persona": "persona", "individuo": "persona",
            "comercio_pequeno": "comercio_pequeno", "comercio pequeno": "comercio_pequeno", "pyme": "comercio_pequeno",
            "comercio_grande": "comercio_grande", "comercio grande": "comercio_grande", "empresa": "comercio_grande"}
CANAL = {"app": "app", "movil": "app", "web": "web", "pos": "pos", "punto_venta": "pos", "api": "api"}
TIPO_DISP = {"android": "android", "ios": "ios", "web": "web", "pos": "pos"}
TIPO_LN = {"ip": "ip", "dispositivo": "dispositivo", "device": "dispositivo", "comercio": "comercio",
           "merchant": "comercio", "tarjeta": "tarjeta", "card": "tarjeta"}
DESECHABLES = {"mailinator.com", "guerrillamail.com", "tempmail.io", "10minutemail.com"}

def mapear(s, mapa, nombre=""):
    k = s.map(clave)
    out = k.map(mapa)
    sin_mapa = s[k.notna() & out.isna()].unique()
    assert len(sin_mapa) == 0, f"{nombre}: valores sin mapear {list(sin_mapa)}"
    return out

def a_bool(s, nombre=""):
    k = s.map(clave)
    out = k.map(BOOL)
    sin_mapa = s[k.notna() & out.isna()].unique()
    assert len(sin_mapa) == 0, f"{nombre}: valores sin mapear {list(sin_mapa)}"
    return out.astype(float)

def parse_fecha(s):
    # fechas de calendario (usuarios, lista negra): 3 formatos, día primero
    s = s.astype("string")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for patron, fmt in [(r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"), (r"^\d{2}/\d{2}/\d{4}$", "%d/%m/%Y"),
                        (r"^\d{2}-\d{2}-\d{4}$", "%d-%m-%Y")]:
        m = s.str.match(patron, na=False)
        out[m] = pd.to_datetime(s[m], format=fmt, errors="coerce")
    return out

OFFSET_CHILE_H = 4   # hora de Chile = UTC-4 (así viene en los datos, incluso en verano; ver 3.1)

def parse_ts_utc(s, asumir_naive_chile=True):
    # Devuelve el instante en UTC (sin zona adjunta). Las fechas sin zona y las dd/mm/yyyy son hora de Chile.
    s = s.astype("string")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    con_zona = s.str.contains(r"(?:Z|[+-]\d{2}:\d{2})$", na=False)
    barra = s.str.match(r"^\d{2}/\d{2}/\d{4}", na=False)
    naive = ~con_zona & ~barra & s.notna()
    desfase = pd.Timedelta(hours=OFFSET_CHILE_H if asumir_naive_chile else 0)
    if con_zona.any():
        out[con_zona] = pd.to_datetime(s[con_zona], utc=True, format="ISO8601", errors="coerce").dt.tz_localize(None)
    if barra.any():
        out[barra] = pd.to_datetime(s[barra], format="%d/%m/%Y %H:%M", errors="coerce") + desfase
    if naive.any():
        out[naive] = pd.to_datetime(s[naive], format="%Y-%m-%d %H:%M:%S", errors="coerce") + desfase
    return out

UMBRAL_MONEDA = 500   # CLP nunca baja de 901 y USD nunca pasa de 313 (se verifica en 3.2): 500 separa sin ambigüedad

def parse_monto(m):
    # 3 dígitos tras el separador (8.199, $1.265) -> punto de MILES (patrón CLP); 2 dígitos (47.33, 6,36) -> decimal (patrón USD)
    s = m.astype("string").str.strip().str.replace("$", "", regex=False)
    miles = s.str.match(r"^-?\d{1,3}(?:\.\d{3})+$", na=False)
    s2 = s.mask(miles, s.str.replace(".", "", regex=False)).str.replace(",", ".", regex=False)
    return pd.to_numeric(s2, errors="coerce"), miles

def ip_valida(ip):
    if pd.isna(ip):
        return False
    p = str(ip).split(".")
    return len(p) == 4 and all(o.isdigit() and 0 <= int(o) <= 255 for o in p)

def normalizar_transacciones(tx):
    # R2-R6: de la fila cruda a una fila normalizada. Es la MISMA función que usará la API para una transacción en línea.
    tx = tx.reset_index(drop=True)
    o = pd.DataFrame({"transaccion_id": tx["transaccion_id"], "usuario_id": tx["usuario_id"],
                      "comercio_id": tx["comercio_id"], "dispositivo_id": tx["dispositivo_id"]})
    o["ts_utc"] = parse_ts_utc(tx["timestamp"])
    valor, miles = parse_monto(tx["monto"])
    moneda = tx["moneda"].map(lambda v: str(v).strip().upper() if pd.notna(v) else np.nan)
    inferida = moneda.isna()
    moneda = moneda.mask(inferida & (miles | (valor.abs() >= UMBRAL_MONEDA)), "CLP")
    moneda = moneda.mask(moneda.isna(), "USD")           # lo que sigue vacío es un monto chico: USD
    o["monto_valor"] = valor
    o["moneda_final"] = moneda
    o["moneda_inferida"] = inferida
    o["monto_negativo"] = valor < 0
    o["monto_usd"] = np.where(moneda == "CLP", valor.abs() / TASA_CLP_USD, valor.abs())
    o["canal"] = mapear(tx["canal"], CANAL, "canal")
    o["pais_tx"] = mapear(tx["pais_transaccion"], PAIS, "pais_transaccion")
    o["mcc"] = tx["mcc"].astype(int).astype(str)
    # El resultado de la autorización NO existe al decidir (la API no lo recibe); solo sirve para limpiar el histórico (R7)
    o["resultado"] = (mapear(tx["resultado_autorizacion"], {"aprobada": "aprobada", "rechazada": "rechazada", "pendiente": "pendiente"}, "resultado")
                      if "resultado_autorizacion" in tx.columns else np.nan)
    lat, lon = pd.to_numeric(tx["latitud"], errors="coerce"), pd.to_numeric(tx["longitud"], errors="coerce")
    o["coord_nula"] = ((lat == 0) & (lon == 0)) | (lat.abs() > 90) | (lon.abs() > 180) | lat.isna()
    o["ip"] = tx["ip"].where(tx["ip"].map(ip_valida))
    o["ip_ausente_o_invalida"] = (~tx["ip"].map(ip_valida)).astype(float)
    o["tarjeta_enmascarada"] = tx["tarjeta_enmascarada"]       # solo para la regla de lista negra; el modelo NUNCA la ve (RNF5)
    o["intentos_previos_1h"] = tx["intentos_previos_1h"].astype(float)
    return o

def normalizar_usuarios(u):
    u = u.drop_duplicates(subset="usuario_id").copy()
    o = pd.DataFrame({"usuario_id": u["usuario_id"]})
    o["pais_usuario"] = mapear(u["pais"], PAIS, "usuarios.pais")
    o["segmento"] = mapear(u["segmento"], SEGMENTO, "usuarios.segmento")
    o["verificado"] = a_bool(u["verificado"], "verificado")
    o["telefono_verificado"] = u["telefono_verificado"].astype(float)
    kyc = u["nivel_kyc"].astype(float)
    o["nivel_kyc"] = kyc.where(kyc.between(0, 3))                                   # R9: -1 -> nulo
    o["dominio_desechable"] = np.where(u["email_dominio"].isna(), np.nan, u["email_dominio"].isin(DESECHABLES).astype(float))
    o["fecha_registro"] = parse_fecha(u["fecha_registro"])
    o["snap_transacciones_previas"] = u["transacciones_previas"].astype(float)       # foto a la fecha de corte: NO se usa (ver 4)
    return o.reset_index(drop=True)

def normalizar_dispositivos(d):
    d = d.drop_duplicates(subset="dispositivo_id")
    o = pd.DataFrame({"dispositivo_id": d["dispositivo_id"]})
    o["tipo_dispositivo"] = mapear(d["tipo"], TIPO_DISP, "dispositivos.tipo")
    o["pais_disp"] = mapear(d["pais_declarado"], PAIS, "pais_declarado")
    o["emulador"] = d["emulador"].astype(float)
    o["snap_usuarios_distintos"] = d["usuarios_distintos"].astype(float)             # foto a la fecha de corte: NO se usa (ver 4)
    return o.reset_index(drop=True)

def normalizar_lista_negra(ln):
    o = pd.DataFrame({"tipo": mapear(ln["tipo"], TIPO_LN, "lista_negra.tipo"), "valor": ln["valor"].str.strip(),
                      "fecha_alta": parse_fecha(ln["fecha_alta"]), "motivo": ln["motivo"], "vigente": ln["vigente"]})
    o["activa"] = o["vigente"] != 0                                    # nulo = se asume activa; 0 = dada de baja
    o["dura"] = (o["vigente"] == 1) & (o["motivo"] == "solicitud_cumplimiento")   # SUPUESTO: solo los pedidos de cumplimiento son regla normativa
    return o.reset_index(drop=True)

print("Funciones de normalización definidas")

### 3.1 R3 · ¿En qué zona horaria están las fechas «sin zona»? (verificado con los datos)

El caso dice «unas en UTC y otras en hora de Chile», pero no dice cuáles. Hay dos formas de comprobarlo:

1. **Pares de la misma transacción** escrita con dos formatos distintos: si la hipótesis es correcta, la diferencia entre ambas debe ser de segundos, no de 4 horas.
2. **Hora del fraude**: el caso afirma que el 49% del fraude confirmado ocurre entre las 00:00 y las 06:00 *(hora de Chile)*.

In [ ]:
t0 = transacciones_raw.drop_duplicates(subset="transaccion_id").reset_index(drop=True)
t0["fraude"] = t0["es_fraude"].map(clave).map(BOOL)
for nombre, naive_chile in [("A · sin zona = hora de Chile (UTC-4)", True), ("B · sin zona = UTC", False)]:
    ts = parse_ts_utc(t0["timestamp"], asumir_naive_chile=naive_chile)
    h_chile = (ts - pd.Timedelta(hours=OFFSET_CHILE_H)).dt.hour            # hora de Chile
    madrugada_fraude = (h_chile[t0["fraude"] == 1] < 6).mean()
    # Pares: misma transacción repetida con formato distinto (usuario, comercio, dispositivo y monto idénticos)
    c = t0.assign(ts=ts, fmt=np.where(t0["timestamp"].str.contains(r"(?:Z|[+-]\d{2}:\d{2})$"), "con zona", "sin zona"))
    c = c.sort_values(["usuario_id", "comercio_id", "monto", "ts"])
    g = c.groupby(["usuario_id", "comercio_id", "dispositivo_id", "monto"], dropna=False)
    c["dif_s"] = g["ts"].diff().dt.total_seconds().abs()
    c["fmt_prev"] = g["fmt"].shift()
    mixtos = c[c["dif_s"].notna() & (c["fmt"] != c["fmt_prev"])]
    print(f"{nombre}\n   fraude entre 00:00 y 06:00: {madrugada_fraude:.1%}   |   pares de formato mixto: {len(mixtos)}, "
          f"diferencia mediana {mixtos['dif_s'].median():,.0f} s")
print("\nEl caso dice 49%. La hipótesis A calza con el caso y deja los pares mixtos separados por segundos; la B los separa por ~4 horas.")

**Resultado:** las fechas sin zona y las `dd/mm/yyyy` están en **hora de Chile con desfase fijo de -4 horas** (incluso en verano, cuando Chile usa UTC-3: es como las escribe el sistema fuente). La regla R3 las convierte a UTC. Convención de aquí en adelante: **`ts_utc` para ordenar y calcular, y la hora local (UTC-4) para variables como «madrugada»** y para el turno de los analistas.

### 3.2 R4 · La moneda vacía se puede inferir

In [ ]:
v, miles = parse_monto(t0["monto"])
mon = t0["moneda"]
print(f"USD: monto máximo {v[mon == 'USD'].abs().max():.0f}   |   CLP: monto mínimo {v[(mon == 'CLP') & (v.abs() > 0)].abs().min():.0f}")
print("→ No hay traslape: un monto ≥ 500 es CLP y uno < 500 es USD. Con punto de miles (3 dígitos tras el punto) también es CLP.")
print(f"Filas sin moneda: {mon.isna().sum():,} ({mon.isna().mean():.1%})")

### 3.3 Aplicar las reglas

Orden: primero R1 (copias exactas), luego normalizar (R2–R6), luego los casi-duplicados (R7), y recién ahí se arma la etiqueta (R8).

In [ ]:
registro = {"filas_entrada": len(transacciones_raw)}

# R1 · copias exactas por transaccion_id
tx_r1 = transacciones_raw.drop_duplicates(subset="transaccion_id").reset_index(drop=True)
registro["R1_copias_exactas_descartadas"] = len(transacciones_raw) - len(tx_r1)

# R2-R6 · normalizar
tx_n = normalizar_transacciones(tx_r1)
registro["R3_fechas_sin_zona_convertidas"] = int((~tx_r1["timestamp"].str.contains(r"(?:Z|[+-]\d{2}:\d{2})$")).sum())
registro["R4_moneda_inferida"] = int(tx_n["moneda_inferida"].sum())
registro["R4_montos_negativos_marcados"] = int(tx_n["monto_negativo"].sum())
registro["R5_coordenadas_nulas"] = int(tx_n["coord_nula"].sum())
registro["R6_ip_ausente_o_invalida"] = int(tx_n["ip_ausente_o_invalida"].sum())

# R7 · casi-duplicados: mismo usuario, comercio, dispositivo, monto y resultado, a ≤ 180 s del anterior
VENTANA_S = 180
tx_n["fraude_bruto"] = tx_r1["es_fraude"].map(clave).map(BOOL).values
orden = tx_n.sort_values(["usuario_id", "comercio_id", "dispositivo_id", "monto_valor", "resultado", "ts_utc"]).index
tmp = tx_n.loc[orden]
g = tmp.groupby(["usuario_id", "comercio_id", "dispositivo_id", "monto_valor", "resultado"], dropna=False)
repetida = (g["ts_utc"].diff().dt.total_seconds() <= VENTANA_S).reindex(tx_n.index).fillna(False)
tx_n["es_duplicado_sistema"] = repetida & (tx_n["resultado"] == "aprobada")          # R7a: se descarta
tx_n["es_reintento"] = repetida & (tx_n["resultado"] != "aprobada")                  # R7b: se conserva y se marca
registro["R7a_duplicados_del_sistema_descartados"] = int(tx_n["es_duplicado_sistema"].sum())
registro["R7b_reintentos_conservados"] = int(tx_n["es_reintento"].sum())
tx_n = tx_n[~tx_n["es_duplicado_sistema"]].reset_index(drop=True)

# R8 · etiqueta de tres estados: 1 fraude, 0 legítima confirmada, NaN sin etiqueta
tx_n["y_conocida"] = tx_n["fraude_bruto"]
tx_n["etiquetada"] = tx_n["y_conocida"].notna()
tx_n["y"] = tx_n["y_conocida"].fillna(0).astype(int)        # para métricas: no revisada = legítima (ver 7.2)
registro["filas_salida"] = len(tx_n)
registro["fraudes_confirmados"] = int(tx_n["y"].sum())
registro["legitimas_confirmadas"] = int((tx_n["y_conocida"] == 0).sum())
registro["sin_etiqueta"] = int((~tx_n["etiquetada"]).sum())
pd.Series(registro, name="cifra")

> **R7 en una línea:** un pago *aprobado* dos veces idéntico en menos de 3 minutos es un duplicado del sistema (un cliente no repite un pago que ya pasó): se borra el segundo. Un pago *rechazado* repetido es un reintento real: **se conserva**, porque es exactamente la señal de velocidad que busca RF10. La ventana de 180 s sale de los datos (los pares se separan por menos de 150 s).

In [ ]:
# Los datasets auxiliares, normalizados con las mismas reglas
usr = normalizar_usuarios(usuarios_raw)
dsp = normalizar_dispositivos(dispositivos_raw)
lneg = normalizar_lista_negra(lista_negra_raw)
print(f"usuarios: {len(usr):,} (duplicados descartados: {len(usuarios_raw) - len(usr)}) | dispositivos: {len(dsp):,} | lista negra: {len(lneg)} entradas")
print(f"Usuarios con fecha_registro futura: {(usr['fecha_registro'] > pd.Timestamp.now()).sum()} (R9)")

## 4 · Variables: solo lo que existe al momento de decidir

La regla de oro del caso de fraude: **una variable entra solo si, en producción, el core de pagos puede entregarla dentro de los 150 ms**. Revisamos cada columna con esa pregunta:

| Columna / idea | ¿Existe al decidir? | Decisión |
|---|---|---|
| Monto (en USD), moneda, canal, MCC, país de la transacción, hora local | Sí, viene en la transacción | **Entra** |
| `intentos_previos_1h` | Sí: lo calcula el core sobre **todo** el tráfico | **Entra** (es la variable de velocidad, RF10) |
| Nivel KYC, verificado, teléfono verificado, dominio de correo desechable, segmento | Sí, del perfil | **Entra** |
| Edad de la cuenta **al momento de la transacción** | Se calcula: `fecha de la tx − fecha_registro` | **Entra** (recalculada) |
| `edad_cuenta_dias` del CSV | **No**: es una foto a la fecha de corte de la muestra | Se descarta y se recalcula |
| Tipo de dispositivo, `emulador`, país declarado del dispositivo | Sí, del tag del cliente | **Entra** |
| `usuarios_distintos` del dispositivo, `transacciones_previas` del usuario | **No**: son conteos sobre todo el histórico, incluidas transacciones *posteriores* | Fuera del modelo (se mide su efecto en 7.3) |
| Coincidencia en lista negra | Sí, pero **solo si la entrada ya existía** en la fecha de la transacción | Regla dura o variable, según el motivo (ver 4.3) |
| `tarjeta_enmascarada` | Es un identificador, no una característica | **No entra al modelo** (RNF5); solo la regla de lista negra la consulta |
| `revisado_por_analista`, `motivo_bloqueo`, `es_fraude` | **No**: llegan después | Fuera (sección 2.2) |
| Velocidad recalculada con la muestra | Engañosa | No se usa (ver 4.2) |

### 4.1 Construir las variables

In [ ]:
COL_LN = {"ip": "ip", "dispositivo": "dispositivo_id", "comercio": "comercio_id", "tarjeta": "tarjeta_enmascarada"}

def aplicar_lista_negra(tx, ln):
    # Coincidencia "puntual en el tiempo": solo cuenta si la entrada ya estaba dada de alta el día de la transacción.
    base = pd.DataFrame({"_i": np.arange(len(tx)), "dia": tx["ts_utc"].dt.normalize().values})
    dura = np.zeros(len(tx), dtype=bool)
    interna = np.zeros(len(tx), dtype=bool)
    for tipo, col in COL_LN.items():
        e = ln[(ln["tipo"] == tipo) & ln["activa"]][["valor", "fecha_alta", "dura"]]
        m = base.assign(valor=tx[col].values).merge(e, on="valor", how="inner")
        m = m[m["fecha_alta"] <= m["dia"]]
        dura[m.loc[m["dura"], "_i"].unique()] = True
        interna[m.loc[~m["dura"], "_i"].unique()] = True
    return dura, interna

FEATURES_NUM = ["log_monto_usd", "hora_local", "es_madrugada", "intentos_previos_1h", "nivel_kyc", "verificado",
                "telefono_verificado", "dominio_desechable", "edad_cuenta_dias", "emulador",
                "pais_distinto_usuario", "pais_distinto_dispositivo", "ip_ausente_o_invalida",
                "dispositivo_ausente", "lista_negra_interna"]
FEATURES_CAT = ["moneda_final", "canal", "pais_tx", "segmento", "tipo_dispositivo", "mcc"]
FEATURES = FEATURES_NUM + FEATURES_CAT

def construir_variables(tx_norm, usr, dsp, ln):
    x = tx_norm.merge(usr.drop(columns=["snap_transacciones_previas"]), on="usuario_id", how="left")
    x = x.merge(dsp.drop(columns=["snap_usuarios_distintos"]), on="dispositivo_id", how="left")
    x["hora_local"] = (x["ts_utc"] - pd.Timedelta(hours=OFFSET_CHILE_H)).dt.hour.astype(float)
    x["es_madrugada"] = (x["hora_local"] < 6).astype(float)
    x["en_turno"] = (x["hora_local"] >= TURNO[0]) & (x["hora_local"] < TURNO[1])
    x["log_monto_usd"] = np.log1p(x["monto_usd"])
    edad = (x["ts_utc"] - x["fecha_registro"]).dt.days.astype(float)
    x["edad_cuenta_dias"] = edad.where(edad >= 0)            # registro posterior a la transacción = imposible -> nulo
    x["pais_distinto_usuario"] = np.where(x["pais_usuario"].isna(), np.nan, (x["pais_tx"] != x["pais_usuario"]).astype(float))
    x["pais_distinto_dispositivo"] = np.where(x["pais_disp"].isna(), np.nan, (x["pais_tx"] != x["pais_disp"]).astype(float))
    x["dispositivo_ausente"] = x["dispositivo_id"].isna().astype(float)
    dura, interna = aplicar_lista_negra(x, ln)
    x["lista_negra_dura"] = dura                              # regla dura: va por encima del modelo (RF4)
    x["lista_negra_interna"] = interna.astype(float)          # señal blanda: el modelo decide cuánto pesa
    for c in FEATURES_CAT:                                    # el imputador espera NaN, no <NA>
        x[c] = x[c].astype(object).where(x[c].notna(), np.nan)
    return x

X_all = construir_variables(tx_n, usr, dsp, lneg)
# Fotos a la fecha de corte: solo para medir la fuga en 7.3, nunca como variable del modelo
snap = tx_n[["usuario_id", "dispositivo_id"]].merge(usr[["usuario_id", "snap_transacciones_previas"]], on="usuario_id", how="left") \
                                              .merge(dsp[["dispositivo_id", "snap_usuarios_distintos"]], on="dispositivo_id", how="left")
X_all["snap_transacciones_previas"] = snap["snap_transacciones_previas"].values
X_all["snap_usuarios_distintos"] = snap["snap_usuarios_distintos"].values
assert len(X_all) == len(tx_n)

perfil = pd.DataFrame({"% nulos": X_all[FEATURES].isna().mean().mul(100).round(1),
                       "únicos": X_all[FEATURES].nunique()})
perfil

Los nulos **no se rellenan aquí**. El imputador del preprocesador (sección 6) aprende las medianas solo con entrenamiento y **viaja con el modelo**: la API recibirá usuarios o dispositivos desconocidos y debe saber qué hacer. Además, el preprocesador agrega un indicador «el dato faltaba», porque que falte puede ser en sí una señal.

### 4.2 Por qué no se recalcula la velocidad con la muestra
RF10 pide «cinco transacciones en dos minutos». Es tentador contarlas en el CSV, pero **la muestra está estratificada**: del tráfico normal quedó una fracción mínima y casi todo el fraude, así que los vecinos de una transacción legítima casi nunca están en el archivo. El conteo saldría distinto para fraude y legítimas por *cómo se armó la muestra*, no por cómo se comporta el cliente. La columna `intentos_previos_1h` la calcula el core sobre el tráfico completo: esa sí vale.

In [ ]:
o = X_all.sort_values("ts_utc")
vecinos = np.zeros(len(o))
pos = {i: k for k, i in enumerate(o.index)}
for _, g in o.groupby("usuario_id"):
    ts = g["ts_utc"].values
    if len(ts) > 1:
        previos = np.arange(len(ts)) - np.searchsorted(ts, ts - np.timedelta64(1, "h"), side="left")
        for i, v in zip(g.index, previos):
            vecinos[pos[i]] = v
o = o.assign(vecinos_en_muestra_1h=vecinos)
comparacion = o.groupby(o["y_conocida"].map({1.0: "fraude", 0.0: "legítima confirmada"}).fillna("sin etiqueta")).agg(
    vecinos_recalculados_en_la_muestra=("vecinos_en_muestra_1h", "mean"),
    intentos_previos_1h_del_core=("intentos_previos_1h", "mean")).round(3)
comparacion

El conteo recalculado en la muestra es prácticamente cero para todos los grupos (no distingue nada); el del core separa claramente el fraude (~5 intentos) de lo legítimo (~0,7). Conclusión: **la velocidad la entrega el core, y en producción vivirá en un almacén de baja latencia (no en Redshift, RT7)**.

### 4.3 Lista negra: regla dura vs señal, y siempre «puntual en el tiempo»
Una entrada con `fecha_alta` posterior a la transacción **no existía** cuando había que decidir: usarla filtra el futuro. Además, no todas las entradas son iguales:

- **Regla dura** (RF4: va por encima de cualquier modelo): entradas vigentes con motivo `solicitud_cumplimiento`. *SUPUESTO a validar con Cumplimiento (Carolina Herrera)*: solo estas son obligación normativa.
- **Señal interna**: el resto de las coincidencias activas (editadas por la célula de fraude). Entran al modelo como una variable más; el modelo aprende cuánto valen.
- Las entradas dadas de baja (`vigente = 0`) no se consideran.
- Las **listas de sanciones internacionales** (RF8) no vienen en la muestra; se integran en la regla dura en la implementación.

In [ ]:
c1 = pd.DataFrame({
    "coincidencias": [int(X_all["lista_negra_dura"].sum()), int((X_all["lista_negra_interna"] == 1).sum())],
    "etiquetadas": [int((X_all["lista_negra_dura"] & X_all["etiquetada"]).sum()), int(((X_all["lista_negra_interna"] == 1) & X_all["etiquetada"]).sum())],
    "fraudes": [int((X_all["lista_negra_dura"] & (X_all["y_conocida"] == 1)).sum()), int(((X_all["lista_negra_interna"] == 1) & (X_all["y_conocida"] == 1)).sum())],
}, index=["regla dura (cumplimiento)", "señal interna"])
print(c1)
sin_pit = 0
for tipo, col in COL_LN.items():
    e = lneg[(lneg["tipo"] == tipo) & lneg["activa"]]
    m = X_all[["ts_utc", col]].assign(dia=X_all["ts_utc"].dt.normalize()).merge(e[["valor", "fecha_alta"]], left_on=col, right_on="valor")
    sin_pit += int((m["fecha_alta"] > m["dia"]).sum())
print(f"\nCoincidencias que SOLO existirían mirando el futuro (entrada dada de alta después de la transacción): {sin_pit}")
print(f"La regla dura marca {X_all['lista_negra_dura'].mean():.2%} de las filas de la muestra.")
print("\nRegla dura, filas que coinciden según el tipo de entrada:")
for tipo, col in COL_LN.items():
    e = lneg[(lneg["tipo"] == tipo) & lneg["activa"] & lneg["dura"]][["valor", "fecha_alta"]]
    mm = X_all[["ts_utc", col, "y", "etiquetada"]].assign(dia=X_all["ts_utc"].dt.normalize()).merge(e, left_on=col, right_on="valor")
    mm = mm[mm["fecha_alta"] <= mm["dia"]]
    print(f"   {tipo:<12} entradas duras: {len(e):>2} · filas que coinciden: {len(mm):>3} · etiquetadas: {int(mm['etiquetada'].sum()):>2} · fraudes: {int(mm['y'].sum())}")

## 5 · Separación en tres conjuntos: **por fecha**, no al azar

En el notebook base se usó un reparto aleatorio estratificado. Aquí **no**: el fraude se adapta (R5) y en producción el modelo siempre predice el *futuro* con datos del *pasado*. Un reparto al azar mezclaría semanas y daría una nota inflada. Se ordena por `ts_utc`:

| Conjunto | Tramo | Para qué |
|---|---|---|
| **Entrenamiento** | primer 60% de la línea de tiempo | el modelo aprende |
| **Validación** | siguiente 20% | decidimos: calibración, umbrales, épocas |
| **Prueba** | último 20% | se informa, **una sola vez**. Es el «conjunto de prueba fijo» de RF11: sus IDs se guardan |

> Limitación honesta: hay solo ~140 fraudes en total, así que validación y prueba tienen decenas. Todo número de prueba lleva intervalo de confianza (sección 10).

In [ ]:
X_all = X_all.sort_values("ts_utc").reset_index(drop=True)
n = len(X_all)
i1, i2 = int(n * 0.60), int(n * 0.80)
tr, va, te = X_all.iloc[:i1].copy(), X_all.iloc[i1:i2].copy(), X_all.iloc[i2:].copy()

for nombre, d in {"Entrenamiento": tr, "Validación": va, "Prueba": te}.items():
    print(f"{nombre:<14} {len(d):>6,} filas · {d['ts_utc'].min():%Y-%m-%d} → {d['ts_utc'].max():%Y-%m-%d} · "
          f"{int(d['y'].sum()):>3} fraudes ({d['y'].mean():.2%}) · {int(d['etiquetada'].sum())} etiquetadas")

y_train, y_val, y_test = tr["y"].to_numpy(), va["y"].to_numpy(), te["y"].to_numpy()
usd_val, usd_test = va["monto_usd"].to_numpy(), te["monto_usd"].to_numpy()
hora_val, hora_test = va["hora_local"].to_numpy(), te["hora_local"].to_numpy()
dura_val, dura_test = va["lista_negra_dura"].to_numpy(), te["lista_negra_dura"].to_numpy()

# Conjunto de prueba fijo (RF11): se guardan sus IDs para comparar siempre contra los mismos datos
RUTA_MODELO.mkdir(parents=True, exist_ok=True)
te[["transaccion_id"]].to_csv(RUTA_MODELO / "test_fijo_ids.csv", index=False)

# Pesos de entrenamiento: el fraude pesa más; las filas sin etiqueta pesan PESO_NO_REVISADAS
def pesos_entrenamiento(d, peso_no_rev=PESO_NO_REVISADAS):
    return np.where(d["y"] == 1, PESO_FRAUDE, np.where(d["etiquetada"], 1.0, peso_no_rev)).astype(np.float32)

w_train = pesos_entrenamiento(tr)
print(f"\nMonto promedio de un fraude en la muestra: USD {tr.loc[tr['y'] == 1, 'monto_usd'].mean():.0f} (el caso dice USD 57)")

## 6 · De la muestra a producción: el número que más se equivoca si se ignora

La muestra tiene ~0,5% de fraude; producción, 0,012%: **~40 veces menos**. Un modelo entrenado en la muestra sobreestima la probabilidad de fraude de cada transacción, y cualquier conteo de bloqueos, revisiones o falsos positivos calculado sobre el archivo queda mal por dos órdenes de magnitud. Se corrige en dos lugares:

1. **Métricas**: cada fila legítima de la muestra *representa* muchas transacciones legítimas de producción. En vez de contar filas, contamos **transacciones por día de producción** (`pesos_dia`): el fraude suma 24/día, las legítimas ~199.976/día.
2. **Probabilidad**: se calibra con Platt sobre validación y se corrige el *prior* (`calibrador`), de modo que «0,3» signifique 30% *en producción*, no en la muestra.

In [ ]:
def pesos_dia(y):
    # Peso de cada fila en "transacciones por día de producción"
    y = np.asarray(y)
    return np.where(y == 1, FRAUDES_DIA / (y == 1).sum(), LEGITIMAS_DIA / (y == 0).sum())

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))

class Calibrador:
    # Platt (a·logit + b) ajustado en validación + corrección de prior muestra -> producción
    def ajustar(self, p_val, y_val):
        lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_val).reshape(-1, 1), y_val)
        self.a, self.b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
        self.pi_ref = float(np.mean(y_val))          # el prior que Platt dejó incorporado
        self.log_razon_prior = float(np.log(PI_PROD / (1 - PI_PROD)) - np.log(self.pi_ref / (1 - self.pi_ref)))
        return self
    def __call__(self, p):
        z = self.a * logit(p) + self.b + self.log_razon_prior
        return 1 / (1 + np.exp(-z))
    def a_dict(self):
        return {"a": self.a, "b": self.b, "log_razon_prior": self.log_razon_prior, "pi_muestra_validacion": self.pi_ref, "pi_produccion": PI_PROD}

# ---------- Decisión: 0 aprobar · 1 revisar · 2 bloquear ----------
def decidir_zona(p, hora, umb, dura=None):
    p, hora = np.asarray(p), np.asarray(hora)
    turno = (hora >= TURNO[0]) & (hora < TURNO[1])
    ub = np.where(turno, umb["bloquear_dia"], umb["bloquear_noche"])
    ur = np.where(turno, umb["revisar_dia"], np.inf)          # de madrugada nadie revisa: no existe la zona "revisar"
    z = np.where(p >= ub, 2, np.where(p >= ur, 1, 0))
    if dura is not None:
        z = np.where(np.asarray(dura), 2, z)                  # RF4: la regla dura manda sobre el modelo
    return z

def proyectar(y, z, usd, wd):
    # Lo que significaría la política en producción: transacciones/día, USD/año.
    y, z, usd, wd = (np.asarray(a) for a in (y, z, usd, wd))
    f, l = y == 1, y == 0
    bloq_f, bloq_l = wd[f & (z == 2)].sum(), wd[l & (z == 2)].sum()
    rev_f, rev_l = wd[f & (z == 1)].sum(), wd[l & (z == 1)].sum()
    # El fraude aprobado se pierde completo. El revisado (siempre dentro de turno) se pierde solo si el analista no lo confirma.
    perdida = 365 * ((wd * usd)[f & (z == 0)].sum() + (1 - EFECTIVIDAD_REVISION) * (wd * usd)[f & (z == 1)].sum())
    costo = perdida + 365 * COSTO_FP_BLOQUEO * bloq_l + 365 * COSTO_REVISION * (rev_f + rev_l)
    total_f = wd[f].sum()
    return {"bloqueos_dia": bloq_f + bloq_l, "fp_dia": bloq_l, "fp_sobre_bloqueos": bloq_l / (bloq_f + bloq_l) if (bloq_f + bloq_l) > 0 else 0.0,
            "revisiones_dia": rev_f + rev_l, "deteccion": (bloq_f + EFECTIVIDAD_REVISION * rev_f) / total_f if total_f > 0 else np.nan,
            "fraude_no_detectado_usd": perdida, "costo_usd": costo}

wd_val, wd_test = pesos_dia(y_val), pesos_dia(y_test)
print("Pesos: cada fila legítima de validación = %.1f tx/día; cada fraude = %.2f tx/día" % (wd_val[y_val == 0][0], wd_val[y_val == 1][0]))

### 6.1 La vara de medir: dos políticas «tontas»

In [ ]:
def zonas_const(n, z):
    return np.full(n, z)

sin_modelo = proyectar(y_test, zonas_const(len(y_test), 0), usd_test, wd_test)        # aprobar todo
bloquear_todo = proyectar(y_test, zonas_const(len(y_test), 2), usd_test, wd_test)      # bloquear todo
print(f"Aprobar todo:   exactitud {np.mean(y_test == 0):.2%} (¡alta!), fraude detectado 0%, "
      f"fraude no detectado USD {sin_modelo['fraude_no_detectado_usd']:,.0f}/año, costo USD {sin_modelo['costo_usd']:,.0f}/año")
print(f"Bloquear todo:  bloquea {bloquear_todo['bloqueos_dia']:,.0f} tx/día, {bloquear_todo['fp_sobre_bloqueos']:.2%} de los bloqueos son falsos, "
      f"costo USD {bloquear_todo['costo_usd']:,.0f}/año")

Aprobar todo tiene **más de 99% de exactitud** (en la prueba el fraude es 0,8% de las filas; en producción sería 99,988%) y deja pasar USD ~500.000 al año: la exactitud no sirve (la frase de Ignacia Bravo). Todo modelo se compara contra **costo anual en USD**, que suma lo que se pierde por fraude, lo que cuesta bloquear a un cliente bueno y lo que cuesta revisar.

### 6.2 Umbrales: los fija riesgo; el notebook propone y muestra la frontera
Se buscan **tres umbrales** sobre la probabilidad calibrada (la de producción):

- `bloquear_dia` y `revisar_dia`: de 08:00 a 22:00 hay analistas, así que existe la zona intermedia «revisar».
- `bloquear_noche`: de madrugada **no hay zona «revisar»** (R4: se revisaría al día siguiente, con el dinero ya liquidado). Solo aprobar o bloquear.

La búsqueda minimiza el costo anual **en validación** sobre las filas que decide el modelo (las que cubre la regla dura se bloquean por la regla y quedan fuera), sujeta a (1) revisiones diarias ≤ la capacidad actual de los analistas y (2) falsos positivos sobre bloqueos ≤ 15% (la meta del directorio). Si ningún umbral cumple (2), se elige el más conservador y **se dice**.

In [ ]:
GRILLA = 1 / (1 + np.exp(-np.arange(-14, 6.01, 0.25)))       # candidatos en escala logit

def _elegir(candidatos):
    # entre empates de costo, el del medio de la meseta (más estable que el borde)
    return candidatos[len(candidatos) // 2]

def buscar_umbrales(p, y, usd, hora, dura):
    # Las filas que decide la regla dura (RF4) no participan: la regla manda y el modelo no puede cambiarlas.
    p, y, usd, hora, dura = (np.asarray(a) for a in (p, y, usd, hora, dura))
    wd = pesos_dia(y)
    m = ~dura
    p, y, usd, hora, wd = p[m], y[m], usd[m], hora[m], wd[m]
    turno = (hora >= TURNO[0]) & (hora < TURNO[1])
    info = {}
    # ---- noche: solo aprobar / bloquear ----
    pn, yn, un, wn = p[~turno], y[~turno], usd[~turno], wd[~turno]
    cand = []
    for ub in GRILLA:
        r = proyectar(yn, np.where(pn >= ub, 2, 0), un, wn)
        cand.append((r["costo_usd"], ub, r["fp_sobre_bloqueos"]))
    ok = [c for c in cand if c[2] <= FP_OBJETIVO] or [max(cand, key=lambda c: c[1])]
    mejor = min(c[0] for c in ok)
    bloq_noche = _elegir([c[1] for c in ok if c[0] <= mejor + 1e-6])
    info["noche_cumple_fp"] = any(c[2] <= FP_OBJETIVO for c in cand)
    # ---- día: aprobar / revisar / bloquear ----
    pd_, yd, ud, wdd = p[turno], y[turno], usd[turno], wd[turno]
    cand = []
    for ur in GRILLA:
        for ub in GRILLA[GRILLA >= ur]:
            z = np.where(pd_ >= ub, 2, np.where(pd_ >= ur, 1, 0))
            r = proyectar(yd, z, ud, wdd)
            cand.append((r["costo_usd"], ur, ub, r["fp_sobre_bloqueos"], r["revisiones_dia"]))
    ok = [c for c in cand if c[3] <= FP_OBJETIVO and c[4] <= CAPACIDAD_REVISION_DIA]
    info["dia_cumple_restricciones"] = bool(ok)
    if not ok:                                   # ninguna combinación cumple: la más conservadora en falsos positivos
        ok = [c for c in cand if c[4] <= CAPACIDAD_REVISION_DIA]
        ok = [min(ok, key=lambda c: (c[3], c[0]))] if ok else [max(cand, key=lambda c: c[2])]
    mejor = min(c[0] for c in ok)
    elegido = _elegir([c for c in ok if c[0] <= mejor + 1e-6])
    umb = {"revisar_dia": float(elegido[1]), "bloquear_dia": float(elegido[2]), "bloquear_noche": float(bloq_noche)}
    return umb, info

print("Búsqueda de umbrales definida")

## 7 · Modelos clásicos de referencia (scikit-learn)

Antes de la red, dos vara de medir con el **mismo preprocesador** (imputar → escalar → one-hot):
- **Regresión logística**: simple, totalmente explicable. Si la red no le gana, no se justifica.
- **Gradient boosting** (`HistGradientBoosting`): el tipo de modelo que suele detectar mejor el fraude y peor se explica (la tensión 3 del caso). Sirve de *techo* para ver cuánto cuesta ser explicable.

In [ ]:
def construir_preprocesador() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median", add_indicator=True)),   # nulo -> mediana + "faltaba"
                          ("escalar", StandardScaler())]), FEATURES_NUM),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="constant", fill_value="desconocido")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), FEATURES_CAT),
    ])

prep = construir_preprocesador().fit(tr[FEATURES])            # SOLO con entrenamiento
A_train = prep.transform(tr[FEATURES]).astype(np.float32)
A_val = prep.transform(va[FEATURES]).astype(np.float32)
A_test = prep.transform(te[FEATURES]).astype(np.float32)
NOMBRES_COLS = list(prep.get_feature_names_out())
print("Forma de A_train:", A_train.shape)

def grupos_de(nombres):
    # Cada columna de la matriz pertenece a un "factor" del negocio: la variable, su indicador de nulo o su one-hot
    cats = sorted(FEATURES_CAT, key=len, reverse=True)
    grupos = {}
    for j, nom in enumerate(nombres):
        if nom.startswith("num__"):
            base = nom[5:]
            base = base.split("missingindicator_", 1)[1] if base.startswith("missingindicator_") else base
        else:
            base = next(c for c in cats if nom.startswith(f"cat__{c}_"))
        grupos.setdefault(base, []).append(j)
    return grupos

GRUPOS = grupos_de(NOMBRES_COLS)
print(f"{len(GRUPOS)} factores:", list(GRUPOS))

In [ ]:
RESULTADOS = {}

def evaluar(nombre, p_val, p_test, framework="scikit-learn"):
    cal = Calibrador().ajustar(p_val, y_val)
    pv, pt = cal(p_val), cal(p_test)
    umb, info = buscar_umbrales(pv, y_val, usd_val, hora_val, dura_val)
    m = ~dura_test                                              # filas que decide el modelo
    z = decidir_zona(pt, hora_test, umb)                        # decisión del modelo solo
    z_sis = decidir_zona(pt, hora_test, umb, dura_test)         # decisión del sistema: regla dura + modelo
    r = proyectar(y_test[m], z[m], usd_test[m], wd_test[m])
    r_sis = proyectar(y_test, z_sis, usd_test, wd_test)
    fila = {"modelo": nombre, "costo_anual_usd": round(r["costo_usd"]), "fraude_no_detectado_usd": round(r["fraude_no_detectado_usd"]),
            "fp_sobre_bloqueos": round(r["fp_sobre_bloqueos"], 3), "bloqueos_dia": round(r["bloqueos_dia"]), "revisiones_dia": round(r["revisiones_dia"]),
            "deteccion": round(r["deteccion"], 3), "roc_auc": round(roc_auc_score(y_test, p_test), 4),
            "pr_auc_prod": round(average_precision_score(y_test, p_test, sample_weight=wd_test), 3)}
    return {"fila": fila, "calibrador": cal, "umbrales": umb, "info": info, "p_test_cal": pt, "p_val_cal": pv,
            "zona_test": z, "zona_sistema": z_sis, "r_modelo": r, "r_sistema": r_sis}

def registrar(nombre, p_val, p_test):
    RESULTADOS[nombre] = evaluar(nombre, p_val, p_test)
    tabla = pd.DataFrame([r["fila"] for r in RESULTADOS.values()]).sort_values("costo_anual_usd").reset_index(drop=True)
    display(tabla)
    return RESULTADOS[nombre]

lr = LogisticRegression(max_iter=3000).fit(A_train, y_train, sample_weight=w_train)
res_lr = registrar("1 · LogReg (scikit-learn)", lr.predict_proba(A_val)[:, 1], lr.predict_proba(A_test)[:, 1])

In [ ]:
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEMILLA)
gb.fit(A_train, y_train, sample_weight=w_train)
res_gb = registrar("2 · Gradient boosting (opaco, techo)", gb.predict_proba(A_val)[:, 1], gb.predict_proba(A_test)[:, 1])

**Cómo leer la tabla.** Las columnas de costo, bloqueos y falsos positivos son **del modelo**, sobre las filas que no decide la regla dura (la misma para todos los modelos; su efecto se ve aparte en la sección 10). `costo_anual_usd` es lo que le costaría a Kipu operar con esa política *a volumen de producción* (fraude que pasa + clientes buenos bloqueados + revisiones). `fp_sobre_bloqueos` es la meta del directorio (< 15%). `pr_auc_prod` es la PR-AUC **re-ponderada a la proporción real**: la de la muestra es optimista porque ahí el fraude es 40 veces más frecuente.

### 7.2 ¿Qué pasa con las filas sin etiqueta? (R8)
Tres formas de tratar el 96% de filas que nadie revisó, medidas con el mismo modelo (regresión logística) y el mismo conjunto de prueba:

In [ ]:
def lr_con(mascara, peso_no_rev):
    w = pesos_entrenamiento(tr, peso_no_rev)
    m = LogisticRegression(max_iter=3000).fit(A_train[mascara], y_train[mascara], sample_weight=w[mascara])
    return m.predict_proba(A_val)[:, 1], m.predict_proba(A_test)[:, 1]

todas = np.ones(len(tr), dtype=bool)
variantes = {"solo revisadas (descarta las sin etiqueta)": (tr["etiquetada"].to_numpy(), 1.0),
             "no revisadas = legítimas, peso 0,25": (todas, 0.25),
             "no revisadas = legítimas, peso 1 (usado)": (todas, 1.0)}
filas_72 = []
for nombre, (mask, w_nr) in variantes.items():
    pv, pt = lr_con(mask, w_nr)
    filas_72.append(evaluar(nombre, pv, pt)["fila"])
pd.DataFrame(filas_72).set_index("modelo")[["roc_auc", "pr_auc_prod", "costo_anual_usd", "fp_sobre_bloqueos", "bloqueos_dia"]]

**Lectura.** Mira primero `roc_auc` y `pr_auc_prod` (miden qué tan bien *ordena* el riesgo y son estables) antes que `costo_anual_usd`, que depende de umbrales elegidos con ~20 fraudes de validación y por eso se mueve de forma ruidosa. Entrenar *solo* con revisadas enseña al modelo a distinguir fraude de «legítima que el motor ya había marcado como sospechosa», una población muy particular y distinta del tráfico que verá en producción; usar también las no revisadas, como legítimas, representa mejor ese tráfico. Esa es la elección por defecto. **Riesgo que queda abierto:** parte de ese 96% puede ser fraude no confirmado (R6). Es ruido de etiqueta: se mide en modo sombra y con muestreo de aprobadas para revisión, no se arregla en el notebook.

### 7.3 ¿Cuánto «ayudaría» filtrar el futuro? (y por qué no se hace)
Se agregan las dos fotos a la fecha de corte (`usuarios_distintos` del dispositivo, `transacciones_previas` del usuario) y se mide el efecto:

In [ ]:
def auc_gb(extra):
    cols = FEATURES + extra
    Xa = np.hstack([A_train, tr[extra].fillna(-1).to_numpy(dtype=np.float32)]) if extra else A_train
    Xt = np.hstack([A_test, te[extra].fillna(-1).to_numpy(dtype=np.float32)]) if extra else A_test
    m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEMILLA).fit(Xa, y_train, sample_weight=w_train)
    p = m.predict_proba(Xt)[:, 1]
    return roc_auc_score(y_test, p), average_precision_score(y_test, p, sample_weight=wd_test)

comp = pd.DataFrame({"sin fotos (usado)": auc_gb([]),
                     "con fotos del futuro": auc_gb(["snap_usuarios_distintos", "snap_transacciones_previas"])},
                    index=["ROC-AUC", "PR-AUC (producción)"]).round(3)
comp

Compara las dos columnas. En esta muestra la ganancia es pequeña o ambigua (el ROC sube apenas; la PR-AUC re-ponderada puede incluso bajar), así que **excluirlas casi no cuesta nada**. Se excluyen **por principio, no por resultado**: en producción el conteo disponible al decidir cubre solo el pasado, y un modelo que se acostumbra a ver el futuro se desploma al desplegarse. El dispositivo compartido sí es una buena señal: la recomendación para la implementación es **construir ese conteo en línea** (el mismo almacén de baja latencia de la velocidad, RT7), con corte temporal correcto.

## 8 · Las redes neuronales (TensorFlow / Keras)

Se entrenan **dos** redes, y la comparación entre ellas es la decisión de arquitectura del caso:

| | **Red opaca (MLP 64-32)** | **Red aditiva explicable (NAM)** |
|---|---|---|
| Idea | Todas las variables se mezclan en capas densas | **Una subred pequeña por factor**; el logit es la **suma** de los aportes de cada factor |
| Explicación | Hay que aproximarla después (SHAP, etc.) | **Exacta**: el aporte de cada factor sale del propio modelo |
| ¿Cumple RT8? | Difícil de defender ante un auditor no técnico | Sí: «esta transacción subió X por el monto, Y por la hora…» |
| Interacciones entre factores | Sí | No (a cambio de ser auditable) |

```
 vector de entrada ─┬─► [subred: monto]        ─► aporte_monto      ┐
                    ├─► [subred: hora]         ─► aporte_hora       ├─► Σ ─► sigmoide ─► probabilidad
                    ├─► [subred: intentos 1h]  ─► aporte_intentos   │
                    └─► ...  (un factor = una subred)               ┘
                        el modelo exporta TAMBIÉN los aportes (RF2, RNF6)
```
Las capas «selectoras» (fijas, no entrenables) reparten las columnas del vector entre las subredes; así el modelo es un solo grafo que se exporta limpio a ONNX.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

def entrenar_tf(tipo="nam", capas=(16, 8), dropout=0.2, l2=1e-4, epocas=100, lr=3e-3, batch=256):
    import keras
    import tensorflow as tf

    keras.utils.set_random_seed(SEMILLA)
    n_in = A_train.shape[1]
    entrada = keras.Input(shape=(n_in,), name="x")
    contribs = None

    if tipo == "mlp":
        h = entrada
        for u in capas:
            h = keras.layers.Dense(u, activation="relu")(h)
            h = keras.layers.Dropout(dropout)(h)
        logit = keras.layers.Dense(1, name="logit")(h)
    else:
        salidas = []
        for nombre, idx in GRUPOS.items():
            sel = np.zeros((n_in, len(idx)), dtype="float32")        # matriz 0/1 que "recorta" las columnas del factor
            for k, j in enumerate(idx):
                sel[j, k] = 1.0
            capa_sel = keras.layers.Dense(len(idx), use_bias=False, trainable=False, name=f"sel_{nombre}")
            h = capa_sel(entrada)
            capa_sel.set_weights([sel])
            for i, u in enumerate(capas):
                h = keras.layers.Dense(u, activation="relu", kernel_regularizer=keras.regularizers.l2(l2), name=f"f_{nombre}_{i}")(h)
            salidas.append(keras.layers.Dense(1, name=f"aporte_{nombre}")(h))
        contribs = keras.layers.Concatenate(name="aportes")(salidas)          # (n, factores)
        capa_suma = keras.layers.Dense(1, use_bias=False, trainable=False, name="suma")
        logit = capa_suma(contribs)
        capa_suma.set_weights([np.ones((len(GRUPOS), 1), dtype="float32")])  # logit = Σ aportes

    prob = keras.layers.Activation("sigmoid", name="prob")(logit)
    modelo = keras.Model(entrada, prob)
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy")
    expl = keras.Model(entrada, [prob, contribs]) if contribs is not None else modelo

    def predecir(A):
        return modelo.predict(np.asarray(A, dtype="float32"), verbose=0, batch_size=4096).ravel()

    def aportes(A):
        if contribs is None:
            return None
        return np.asarray(expl.predict(np.asarray(A, dtype="float32"), verbose=0, batch_size=4096)[1])

    y_tr = y_train.astype("float32")
    mejor_ap, mejores_pesos, hechas = -1.0, None, 0
    while hechas < epocas:                                   # de a 10 épocas, como en el notebook base
        tramo = min(10, epocas - hechas)
        modelo.fit(A_train, y_tr, sample_weight=w_train, epochs=tramo, batch_size=batch, verbose=0)
        hechas += tramo
        ap = average_precision_score(y_val, predecir(A_val), sample_weight=wd_val)
        print(f"época {hechas:>3} · PR-AUC (producción) en validación {ap:.3f}")
        if ap > mejor_ap:
            mejor_ap, mejores_pesos = ap, modelo.get_weights()
    modelo.set_weights(mejores_pesos)                        # nos quedamos con la mejor época según VALIDACIÓN

    def exportar_onnx(ruta):
        try:
            expl.export(str(ruta), format="onnx")            # Keras 3 usa tf2onnx por debajo
        except Exception as e:                               # plan B: convertir directamente con tf2onnx
            print("export(format='onnx') falló:", repr(e)[:120], "→ intento con tf2onnx")
            import tf2onnx
            spec = (tf.TensorSpec((None, n_in), tf.float32, name="x"),)
            tf2onnx.convert.from_keras(expl, input_signature=spec, opset=17, output_path=str(ruta))

    return {"predecir": predecir, "aportes": aportes, "exportar_onnx": exportar_onnx, "mejor_epoca_pr_auc": mejor_ap,
            "framework": f"tensorflow {tf.__version__}", "tipo": tipo}

In [ ]:
t0 = time.perf_counter()
red_mlp = entrenar_tf("mlp", capas=(64, 32), dropout=0.2, epocas=60, lr=1e-3)
print(f"Entrenada con {red_mlp['framework']} en {time.perf_counter() - t0:.1f} s")
res_mlp = registrar("3 · Red MLP 64-32 (opaca)", red_mlp["predecir"](A_val), red_mlp["predecir"](A_test))

In [ ]:
t0 = time.perf_counter()
red_nam = entrenar_tf("nam", capas=(16, 8), l2=1e-4, epocas=100, lr=3e-3)
print(f"Entrenada con {red_nam['framework']} en {time.perf_counter() - t0:.1f} s")
res_nam = registrar("4 · Red aditiva NAM (explicable)", red_nam["predecir"](A_val), red_nam["predecir"](A_test))

**Qué buscar en la tabla:**
- ¿Cuánto cuesta (en USD/año y en PR-AUC) ser explicable? Si la NAM queda cerca del gradient boosting y de la red opaca, **la explicabilidad es casi gratis** y el argumento de «el modelo que mejor detecta es el que peor se explica» pierde fuerza en este problema.
- ¿Gana la NAM a la regresión logística? Si no, no se justifica una red.
- Con solo decenas de fraudes en prueba, **diferencias pequeñas entre modelos no son diferencias**: mira los intervalos de la sección 10 antes de declarar un ganador.

El modelo candidato a producción es la **NAM**, salvo que el gate de la sección 12 diga otra cosa.

## 9 · Evaluación del candidato (NAM)

### 9.1 ¿Separa el modelo el fraude de lo legítimo?

In [ ]:
cand = res_nam
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

from sklearn.metrics import precision_recall_curve, roc_curve
for nombre, r in RESULTADOS.items():
    fpr, tpr, _ = roc_curve(y_test, r["p_test_cal"])
    ax[0].plot(fpr, tpr, label=nombre.split("(")[0].strip())
    pr, rc, _ = precision_recall_curve(y_test, r["p_test_cal"], sample_weight=wd_test)
    ax[1].plot(rc, pr, label=nombre.split("(")[0].strip())
ax[0].plot([0, 1], [0, 1], "--", color="gray"); ax[0].set_title("ROC (prueba)"); ax[0].set_xlabel("falsos positivos (tasa)"); ax[0].set_ylabel("fraude detectado")
ax[1].set_title("Precisión vs recall a proporción de PRODUCCIÓN"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precisión"); ax[1].legend(fontsize=7)

bins = np.linspace(-14, 6, 41)
ax[2].hist(logit(cand["p_test_cal"])[y_test == 0], bins=bins, alpha=.6, density=True, label="legítimas")
ax[2].hist(logit(cand["p_test_cal"])[y_test == 1], bins=bins, alpha=.6, density=True, label="fraude")
for k, v in cand["umbrales"].items():
    ax[2].axvline(logit(v), ls="--", lw=1, color="k")
ax[2].set_title("Riesgo calibrado (logit) y umbrales"); ax[2].legend()
plt.tight_layout(); plt.show()
print("Umbrales propuestos (probabilidad de fraude EN PRODUCCIÓN):", {k: f"{v:.4g}" for k, v in cand["umbrales"].items()})
print("¿Cumplen restricciones en validación?", cand["info"])

### 9.2 Calibración: «0,3» tiene que significar 30% en producción
Se agrupan las transacciones de validación por riesgo predicho y se compara con lo observado (re-ponderado a producción):

In [ ]:
pv = cand["p_val_cal"]
cortes = [0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 0.5, 1.0001]
grupo = pd.cut(pv, cortes, right=False)
cal_df = pd.DataFrame({"g": grupo, "w": wd_val, "fraude_w": wd_val * y_val}).groupby("g", observed=True).agg(
    transacciones_dia=("w", "sum"), fraudes_dia=("fraude_w", "sum"))
cal_df["fraude_observado"] = (cal_df["fraudes_dia"] / cal_df["transacciones_dia"]).round(5)
cal_df["prob_media_predicha"] = pd.Series(pv).groupby(grupo, observed=True).mean().round(5).values
cal_df.round(3)

Con solo decenas de fraudes en validación, los tramos altos tienen muy pocos casos: la calibración es una **aproximación**, no una garantía. Por eso el protocolo de la sección 13 repite esta tabla semanalmente con las etiquetas confirmadas (RNF8).

## 10 · ¿Se cumplen las metas del directorio? Con intervalos, no con un punto

In [ ]:
def bootstrap(y, z, usd, wd, B=500, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    y, z, usd, wd = (np.asarray(a) for a in (y, z, usd, wd))
    f, l = np.where(y == 1)[0], np.where(y == 0)[0]
    filas = []
    for _ in range(B):
        idx = np.concatenate([rng.choice(f, len(f)), rng.choice(l, len(l))])
        filas.append(proyectar(y[idx], z[idx], usd[idx], wd[idx]))
    return pd.DataFrame(filas)

z_c = cand["zona_test"]                      # decisión del modelo solo
m_t = ~dura_test                             # filas que decide el modelo (no la regla dura)
yt, zt, ut, wt = y_test[m_t], z_c[m_t], usd_test[m_t], wd_test[m_t]
punto = proyectar(yt, zt, ut, wt)
bs = bootstrap(yt, zt, ut, wt)
def ic(col):
    return f"[{bs[col].quantile(.05):,.3g} – {bs[col].quantile(.95):,.3g}]"

n_f, n_l = int((yt == 1).sum()), int((yt == 0).sum())
fp_filas = int(((zt == 2) & (yt == 0)).sum())
tp_filas = int(((zt == 2) & (yt == 1)).sum())
w_legit = float(wd_test[y_test == 0][0])     # transacciones/día que representa una fila legítima de la muestra
tabla_metas = pd.DataFrame([
    ["Falsos positivos sobre bloqueos", "< 15%", f"{punto['fp_sobre_bloqueos']:.1%}", ic("fp_sobre_bloqueos")],
    ["Fraude no detectado (USD/año)", f"< {FRAUDE_NO_DET_OBJETIVO:,}", f"{punto['fraude_no_detectado_usd']:,.0f}", ic("fraude_no_detectado_usd")],
    ["Revisiones por día", f"≤ {CAPACIDAD_REVISION_DIA:,}", f"{punto['revisiones_dia']:,.0f}", ic("revisiones_dia")],
    ["Bloqueos por día, solo modelo (hoy: 1.150)", "—", f"{punto['bloqueos_dia']:,.0f}", ic("bloqueos_dia")],
    ["Costo anual total (USD)", "—", f"{punto['costo_usd']:,.0f}", ic("costo_usd")],
], columns=["indicador", "meta", "estimado en prueba", "intervalo 90% (bootstrap)"])
tabla_metas

In [ ]:
# La cuenta que el caso obliga a hacer: ¿qué puede demostrar un conjunto de prueba con tan pocos casos?
print(f"Prueba: {n_f} fraudes y {n_l:,} legítimas. Entre los bloqueos hay {tp_filas} fraudes y {fp_filas} legítimas.")
print(f"Cada legítima bloqueada en la muestra equivale a ~{w_legit:.0f} clientes buenos bloqueados POR DÍA en producción.")
if fp_filas == 0:
    cota = 3 * w_legit                                # regla de tres: cota superior al 95% con 0 eventos en n
    tp_dia = tp_filas / n_f * FRAUDES_DIA
    print(f"\nCon 0 falsos positivos observados, la regla de tres solo permite afirmar «hasta ~{cota:,.0f} clientes buenos/día» (95%).")
    print(f"Con {tp_dia:.0f} fraudes bloqueados/día, eso es compatible con una tasa de falsos positivos sobre bloqueos de hasta {cota / (cota + tp_dia):.0%}.")
else:
    print(f"\nUn solo falso positivo más o menos cambia la tasa de falsos positivos sobre bloqueos de forma visible: ver el intervalo.")

### Lo que esto significa para el criterio de éxito 1 (FP < 15%)
Con 0,012% de prevalencia, para que **menos del 15% de los bloqueos sean falsos** hay que bloquear con casi cero error sobre ~200.000 transacciones legítimas por día: una exigencia de varios nueves. **La muestra estratificada puede mostrar que el modelo separa muy bien, pero no puede certificar esa meta**: el intervalo de arriba es ancho porque un solo cliente bueno bloqueado en la muestra pesa decenas por día en producción. La meta se **demuestra en modo sombra** (8 semanas, tráfico real, criterio 1 del caso), y el diseño ayuda a cumplirla: bloquear solo lo muy seguro, mandar lo dudoso a **revisión** (con capacidad de los analistas como límite) y aprobar el resto.

> **Para la defensa:** *«El notebook no afirma que cumplimos la meta de falsos positivos; afirma que el modelo es compatible con ella y explica por qué solo el modo sombra puede confirmarla».* Es una afirmación honesta y verificable; la contraria no lo es.

### 10.0 La regla dura, aparte: lo que pasa cuando el sistema completo incluye RF4
Las métricas de arriba son del **modelo**. El sistema completo agrega la regla dura (RF4), que decide por su cuenta. Con la definición que usamos (*SUPUESTO*: lista negra vigente por `solicitud_cumplimiento`):

In [ ]:
r_sis = proyectar(y_test, cand["zona_sistema"], usd_test, wd_test)
dura_legit_dia = float(wd_test[dura_test & (y_test == 0)].sum())
dura_fraude_dia = float(wd_test[dura_test & (y_test == 1)].sum())
dura_bloq_dia = dura_legit_dia + dura_fraude_dia
print(f"La regla dura marca {int(dura_test.sum())} filas de prueba ({dura_test.mean():.2%}): ~{dura_bloq_dia:,.0f} bloqueos/día en producción, "
      f"de los cuales ~{dura_fraude_dia:.1f}/día son fraude.")
print(f"Sistema completo (regla + modelo): {r_sis['bloqueos_dia']:,.0f} bloqueos/día, {r_sis['fp_sobre_bloqueos']:.1%} de ellos falsos; "
      f"costo USD {r_sis['costo_usd']:,.0f}/año (solo modelo: USD {punto['costo_usd']:,.0f}).")
por_tipo = []
for tipo, col in COL_LN.items():
    e = lneg[(lneg["tipo"] == tipo) & lneg["activa"] & lneg["dura"]][["valor", "fecha_alta"]]
    mm = te[["ts_utc", col, "y"]].assign(dia=te["ts_utc"].dt.normalize()).merge(e, left_on=col, right_on="valor")
    mm = mm[mm["fecha_alta"] <= mm["dia"]]
    por_tipo.append({"tipo de entrada": tipo, "entradas duras": len(e), "filas de prueba que coinciden": len(mm), "de ellas, fraudes": int(mm["y"].sum())})
pd.DataFrame(por_tipo)

**Lectura.** Si la regla dura bloquea del orden de mil o más clientes buenos al día y casi no atrapa fraude, **el problema no es el modelo: es la definición de la regla** (en esta muestra la dominan unos pocos comercios muy activos con una entrada vigente). La regla manda sobre el modelo por exigencia normativa (RF4), así que su definición exacta —¿bloquear *todo* un comercio por una solicitud de cumplimiento? ¿solo las operaciones nuevas?— es una **decisión de Cumplimiento (Carolina Herrera)**, no de ingeniería. Dos cautelas: (1) esto depende de que las filas legítimas de la muestra sean representativas del tráfico legítimo, supuesto estándar pero no verificable aquí; (2) las tarjetas enmascaradas de la muestra **no son únicas** (varias transacciones comparten los últimos 4 dígitos), así que coincidir por tarjeta en la muestra no es confiable; en producción el identificador sustituto sí lo es (RT4).

### 10.1 La frontera costo–riesgo: lo que riesgo necesita para fijar los umbrales (RF3, DA5)
Los umbrales los fija riesgo (DA5). Esta tabla es su insumo: qué pasa si el umbral de bloqueo sube o baja (se mantiene el de revisión propuesto); es la frontera **del modelo**, sin las filas de la regla dura. Se calcula en **validación** (con la que se decidió) y se muestra la prueba al lado, sin usarla para elegir.

In [ ]:
ur = cand["umbrales"]["revisar_dia"]
m_v = ~dura_val
filas = []
for ub in GRILLA[(GRILLA >= 1e-4) & (GRILLA <= 0.99)][::2]:
    umb = {"revisar_dia": min(ur, ub), "bloquear_dia": ub, "bloquear_noche": ub}
    zv = decidir_zona(cand["p_val_cal"], hora_val, umb)
    zt_ = decidir_zona(cand["p_test_cal"], hora_test, umb)
    rv = proyectar(y_val[m_v], zv[m_v], usd_val[m_v], wd_val[m_v])
    rt = proyectar(y_test[m_t], zt_[m_t], usd_test[m_t], wd_test[m_t])
    filas.append({"umbral_bloqueo": ub, "fp_sobre_bloq_val": rv["fp_sobre_bloqueos"], "fp_sobre_bloq_test": rt["fp_sobre_bloqueos"],
                  "fraude_no_det_val": rv["fraude_no_detectado_usd"], "fraude_no_det_test": rt["fraude_no_detectado_usd"],
                  "bloqueos_dia_test": rt["bloqueos_dia"], "costo_val": rv["costo_usd"], "costo_test": rt["costo_usd"]})
frontera = pd.DataFrame(filas)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for sufijo, est in [("val", "-"), ("test", ":")]:
    ax[0].plot(frontera["umbral_bloqueo"], frontera[f"fp_sobre_bloq_{sufijo}"], est, label=sufijo)
    ax[1].plot(frontera["umbral_bloqueo"], frontera[f"fraude_no_det_{sufijo}"], est, label=sufijo)
    ax[2].plot(frontera["umbral_bloqueo"], frontera[f"costo_{sufijo}"], est, label=sufijo)
ax[0].axhline(FP_OBJETIVO, color="r", lw=.8); ax[0].set_title("FP sobre bloqueos (meta 15%)")
ax[1].axhline(FRAUDE_NO_DET_OBJETIVO, color="r", lw=.8); ax[1].set_title("Fraude no detectado USD/año (meta 350.000)")
ax[2].set_title("Costo anual USD")
for a in ax:
    a.set_xscale("log"); a.set_xlabel("umbral de bloqueo (prob. en producción)"); a.legend()
plt.tight_layout(); plt.show()
frontera.round(4).head(12)

## 11 · Explicabilidad: de «el modelo dice 0,8» a «por estas razones»

Gracias a la arquitectura aditiva, el aporte de cada factor es **exacto** y se mide contra un caso «promedio» (el aporte medio en entrenamiento). Un aporte positivo sube el riesgo; negativo, lo baja. Se multiplica por la pendiente de calibración para que los números estén en la misma escala que la probabilidad de producción.

### 11.1 Mirada global: la «forma» que aprendió la red para cada factor
Esto es lo que un auditor puede revisar *sin ver código*: «a más intentos en la última hora, más riesgo».

In [ ]:
BASE_APORTES = red_nam["aportes"](A_train).mean(axis=0)        # aporte medio por factor en entrenamiento
NOMBRES_GRUPOS = list(GRUPOS)
fila_mediana = {c: tr[c].median() for c in FEATURES_NUM}
fila_mediana.update({c: tr[c].mode().iloc[0] for c in FEATURES_CAT})

def curva_factor(factor, valores):
    d = pd.DataFrame([fila_mediana] * len(valores))
    d[factor] = valores
    A = prep.transform(d[FEATURES]).astype(np.float32)
    return (red_nam["aportes"](A)[:, NOMBRES_GRUPOS.index(factor)] - BASE_APORTES[NOMBRES_GRUPOS.index(factor)]) * res_nam["calibrador"].a

fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
for a, (f, vals) in zip(ax, [("log_monto_usd", np.linspace(0, np.log1p(300), 60)), ("hora_local", np.arange(0, 24)),
                             ("intentos_previos_1h", np.arange(0, 13)), ("edad_cuenta_dias", np.linspace(0, 3000, 60))]):
    a.plot(vals, curva_factor(f, vals), marker="o" if len(vals) < 30 else None)
    a.axhline(0, color="gray", lw=.6); a.set_title(f"aporte de «{f}»"); a.set_ylabel("log-odds")
plt.tight_layout(); plt.show()

# Los factores que más pesan en promedio (valor absoluto del aporte, sobre prueba)
ap_test = (red_nam["aportes"](A_test) - BASE_APORTES) * res_nam["calibrador"].a
peso_global = pd.Series(np.abs(ap_test).mean(axis=0), index=NOMBRES_GRUPOS).sort_values(ascending=False)
peso_global.head(10).round(3).to_frame("aporte absoluto medio")

### 11.2 Mirada por transacción: la función de decisión que usará la API

`decidir(...)` recibe **solo lo que sabe el core al autorizar** (nótese que quitamos `resultado_autorizacion`, `es_fraude`, `revisado_por_analista` y `motivo_bloqueo`: no existen todavía) y devuelve decisión, razones y un **registro de auditoría** con versión de modelo, versión de umbrales y valores de entrada (RF7, RNF7). El número de tarjeta no se registra (RNF5).

In [ ]:
TEXTO_FACTOR = {
    "log_monto_usd": "Monto de la transacción", "hora_local": "Hora del día (Chile)", "es_madrugada": "Transacción de madrugada (00–06)",
    "intentos_previos_1h": "Intentos previos en la última hora", "nivel_kyc": "Nivel de verificación (KYC)",
    "verificado": "Usuario verificado", "telefono_verificado": "Teléfono verificado", "dominio_desechable": "Correo de dominio desechable",
    "edad_cuenta_dias": "Antigüedad de la cuenta (días)", "emulador": "Dispositivo emulado",
    "pais_distinto_usuario": "País distinto al del usuario", "pais_distinto_dispositivo": "País distinto al del dispositivo",
    "ip_ausente_o_invalida": "IP ausente o inválida", "dispositivo_ausente": "Dispositivo no identificado",
    "lista_negra_interna": "Coincidencia en lista negra interna", "moneda_final": "Moneda", "canal": "Canal",
    "pais_tx": "País de la transacción", "segmento": "Segmento del usuario", "tipo_dispositivo": "Tipo de dispositivo", "mcc": "Rubro del comercio (MCC)",
}
ZONA_TXT = {0: "aprobar", 1: "revisar", 2: "bloquear"}
CAMPOS_FUTUROS = ["es_fraude", "revisado_por_analista", "motivo_bloqueo", "resultado_autorizacion"]   # no existen al decidir

def a_python(o):
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, dict):
        return {k: a_python(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [a_python(v) for v in o]
    return o

def decidir(registro, scorer, ctx, top=5):
    # registro: dict con los campos crudos de la transacción. ctx: prep, tablas, calibrador, umbrales, aportes base, versiones.
    t0 = time.perf_counter()
    x = construir_variables(normalizar_transacciones(pd.DataFrame([registro])), ctx["usr"], ctx["dsp"], ctx["lneg"])
    A = ctx["prep"].transform(x[FEATURES]).astype(np.float32)
    p_crudo, aportes = scorer(A)
    p = float(ctx["cal"](p_crudo)[0])
    dura = bool(x["lista_negra_dura"].iloc[0])
    zona = int(decidir_zona(p, x["hora_local"].to_numpy(), ctx["umbrales"], np.array([dura]))[0])
    razones = []
    if dura:
        razones.append({"factor": "regla_dura", "descripcion": "Entrada vigente en lista negra por solicitud de cumplimiento", "efecto": "bloqueo normativo"})
    if aportes is not None:
        delta = (aportes[0] - ctx["base_aportes"]) * ctx["cal"].a
        for j in np.argsort(-np.abs(delta))[:top]:
            f = ctx["factores"][j]
            razones.append({"factor": f, "descripcion": TEXTO_FACTOR.get(f, f), "valor": x[f].iloc[0],
                            "aporte_logodds": round(float(delta[j]), 3), "efecto": "sube el riesgo" if delta[j] > 0 else "baja el riesgo"})
    resultado = {"decision": ZONA_TXT[zona], "probabilidad_fraude": p, "razones": razones}
    auditoria = {"transaccion_id": registro.get("transaccion_id"), "ts_utc": str(x["ts_utc"].iloc[0]),
                 "version_modelo": ctx["version_modelo"], "version_umbrales": ctx["version_umbrales"],
                 "entradas": {k: v for k, v in registro.items() if k != "tarjeta_enmascarada"},       # RNF5: la tarjeta no se registra
                 "variables": {f: x[f].iloc[0] for f in FEATURES}, "p_modelo_crudo": float(p_crudo[0]),
                 "umbrales_aplicados": ctx["umbrales"], "regla_dura": dura, **resultado,
                 "latencia_ms": round((time.perf_counter() - t0) * 1000, 2)}
    return a_python(resultado), a_python(auditoria)

crudo_por_id = transacciones_raw.drop_duplicates("transaccion_id").set_index("transaccion_id", drop=False)

def registro_json(tid):
    r = crudo_por_id.loc[tid].drop(labels=CAMPOS_FUTUROS).to_dict()
    return {k: (None if pd.isna(v) else (int(v) if isinstance(v, (np.integer,)) else v)) for k, v in r.items()}

CTX = {"prep": prep, "usr": usr, "dsp": dsp, "lneg": lneg, "cal": res_nam["calibrador"], "umbrales": res_nam["umbrales"],
       "base_aportes": BASE_APORTES, "factores": NOMBRES_GRUPOS, "version_modelo": "2.0.0-borrador", "version_umbrales": "u-0001"}
scorer_keras = lambda A: (red_nam["predecir"](A), red_nam["aportes"](A))

# Una transacción bloqueada de la prueba: lo que vería el analista (RNF6: razones en < 5 s)
ids_bloq = te.loc[res_nam["zona_test"] == 2, "transaccion_id"].tolist()
if ids_bloq:
    tid = ids_bloq[0]
    res, aud = decidir(registro_json(tid), scorer_keras, CTX)
    print(json.dumps(res, indent=2, ensure_ascii=False))
    print(f"\nLatencia de la decisión completa (incluye limpieza + variables): {aud['latencia_ms']} ms")

### 11.3 Reproducibilidad: «un auditor pide una transacción al azar» (criterio de éxito 4)
Con el registro de auditoría guardado se puede **reproducir** la decisión: mismas entradas, misma versión de modelo y umbrales → misma respuesta.

In [ ]:
rng = np.random.default_rng(SEMILLA)
tid = te["transaccion_id"].iloc[int(rng.integers(len(te)))]
res1, aud1 = decidir(registro_json(tid), scorer_keras, CTX)
res2, aud2 = decidir(aud1["entradas"] | {"tarjeta_enmascarada": crudo_por_id.loc[tid, "tarjeta_enmascarada"]}, scorer_keras, CTX)
assert res1 == res2, "La decisión no es reproducible"
print(f"Transacción al azar {tid}: decisión «{res1['decision']}», p = {res1['probabilidad_fraude']:.2e}")
print("Reproducida desde el registro de auditoría → idéntica ✔")
print("Campos del registro:", list(aud1))

## 12 · Puerta de publicación (RF11): no se publica un modelo que empeora

RF11 pide reentrenar y **no publicar si el modelo nuevo empeora contra el vigente en un conjunto de prueba fijo**. Esa puerta vive en el pipeline, no en la cabeza de alguien. Aquí queda escrita: el candidato debe costar **no más** y ordenar **no peor** (PR-AUC en producción, con una tolerancia pequeña) que el modelo vigente, ambos sobre `test_fijo_ids.csv`.

> Mientras no exista un modelo vigente, la vara es la regresión logística. Cuando arranque el modo sombra, el vigente real es el **motor de reglas** y la comparación se hace sobre el mismo tráfico (criterios de éxito 1 y 2).

In [ ]:
TOLERANCIA_PR_AUC = 0.01

def puede_publicar(candidato, vigente):
    c, v = candidato["fila"], vigente["fila"]
    razones = []
    if c["costo_anual_usd"] > v["costo_anual_usd"]:
        razones.append(f"costo {c['costo_anual_usd']:,} > vigente {v['costo_anual_usd']:,}")
    if c["pr_auc_prod"] < v["pr_auc_prod"] - TOLERANCIA_PR_AUC:
        razones.append(f"PR-AUC {c['pr_auc_prod']} < vigente {v['pr_auc_prod']}")
    return len(razones) == 0, razones

ok_pub, motivos = puede_publicar(res_nam, res_lr)
print("Candidato: NAM · Vigente (referencia): regresión logística")
print("→ PUBLICABLE" if ok_pub else "→ NO SE PUBLICA: " + "; ".join(motivos))

## 13 · Exportar a ONNX y verificar

Se exportan **cuatro archivos** a `model/`:

| Archivo | Contenido | Quién lo usa en la API |
|---|---|---|
| `preprocesador.joblib` | Imputar, escalar y one-hot (scikit-learn) | Transforma la transacción en el vector de entrada |
| `modelo.onnx` | La red NAM: **dos salidas**, probabilidad cruda y aportes por factor | `onnxruntime`; el aporte de cada factor sale del mismo cálculo (RF2) |
| `metadata.json` | Versión, framework, calibración, aportes base, huella de los datos, métricas, versión de scikit-learn | Reconstruir la probabilidad de producción y explicar; trazabilidad (RF7) |
| `umbrales.json` | Los tres umbrales, con quién, cuándo y por qué | **Se cambia sin redesplegar** (RF3); cada cambio deja registro (RNF10) |

**Los umbrales no van dentro del modelo.** Si mañana riesgo decide ser más estricto, cambia `umbrales.json` (con autorización, DA5/RO5) y no se toca el modelo ni la imagen. Aquí queda como **propuesta pendiente de autorización de riesgo**.

> Antes de dar por bueno el ONNX se **verifica**: `onnxruntime` debe reproducir la probabilidad y los aportes de la red de Keras.

In [ ]:
import onnxruntime as ort

def salidas_onnx(sesion, A):
    nombre = sesion.get_inputs()[0].name
    outs = sesion.run(None, {nombre: np.asarray(A, dtype=np.float32)})
    p = next(o for o in outs if o.ndim == 2 and o.shape[1] == 1).ravel()
    c = next((o for o in outs if o.ndim == 2 and o.shape[1] > 1), None)
    return p, c

def exportar(red, resultado, version, descripcion, version_umbrales="u-0001"):
    assert ok_pub, "El gate de publicación (RF11) no se cumple: no se exporta"
    RUTA_MODELO.mkdir(parents=True, exist_ok=True)
    for viejo in ["modelo.onnx", "preprocesador.joblib", "metadata.json", "umbrales.json"]:
        (RUTA_MODELO / viejo).unlink(missing_ok=True)
    joblib.dump(prep, RUTA_MODELO / "preprocesador.joblib")
    red["exportar_onnx"](RUTA_MODELO / "modelo.onnx")

    # Verificación: lo que usará la API (onnxruntime) vs. la red original
    sesion = ort.InferenceSession(str(RUTA_MODELO / "modelo.onnx"), providers=["CPUExecutionProvider"])
    p_onnx, c_onnx = salidas_onnx(sesion, A_test)
    dif_p = float(np.abs(p_onnx - red["predecir"](A_test)).max())
    dif_c = float(np.abs(c_onnx - red["aportes"](A_test)).max())
    assert dif_p < 1e-4 and dif_c < 1e-3, f"El ONNX no reproduce a la red (prob {dif_p}, aportes {dif_c})"
    print(f"ONNX verificado: diferencia máxima en probabilidad {dif_p:.1e} y en aportes {dif_c:.1e}")

    cal = resultado["calibrador"]
    r = resultado["fila"]
    metadata = {
        "nombre": "fraude-kipu", "version": version, "formato": "onnx", "framework": red["framework"], "descripcion": descripcion,
        "entrenado_en": datetime.now(timezone.utc).isoformat(timespec="seconds"), "sklearn_version": sklearn.__version__,
        "arquitectura": "NAM: una subred por factor, logit = suma de aportes (explicable por construcción)",
        "variables_numericas": FEATURES_NUM, "variables_categoricas": FEATURES_CAT,
        "factores": NOMBRES_GRUPOS, "aportes_base": [float(v) for v in BASE_APORTES],
        "calibracion": cal.a_dict(),
        "zona_horaria": {"sin_zona_es": "hora de Chile (UTC-4 fijo)", "offset_horas": OFFSET_CHILE_H, "turno_analistas": list(TURNO)},
        "supuestos": {"tasa_clp_usd": TASA_CLP_USD, "costo_revision_usd": COSTO_REVISION, "efectividad_revision": EFECTIVIDAD_REVISION,
                      "costo_fp_bloqueo_usd": round(COSTO_FP_BLOQUEO, 2), "peso_no_revisadas": PESO_NO_REVISADAS, "peso_fraude_entrenamiento": PESO_FRAUDE,
                      "regla_dura": "lista negra vigente con motivo solicitud_cumplimiento"},
        "datos": {"huella_sha256_16": HASH_DATOS, "filas_entrenamiento": int(len(tr)), "fraudes_entrenamiento": int(y_train.sum()),
                  "periodo_entrenamiento": [str(tr["ts_utc"].min()), str(tr["ts_utc"].max())],
                  "periodo_prueba": [str(te["ts_utc"].min()), str(te["ts_utc"].max())], "limpieza": registro},
        "columnas_excluidas_por_fuga": ["revisado_por_analista", "motivo_bloqueo", "es_fraude", "resultado_autorizacion",
                                        "usuarios_distintos", "transacciones_previas", "edad_cuenta_dias (del CSV)"],
        "metricas_prueba": {**{k: v for k, v in r.items() if k != "modelo"},
                            "nota": "proyectadas a producción (0,012% de fraude, 200.000 tx/día); n fraudes en prueba = %d" % int((y_test == 1).sum())},
    }
    (RUTA_MODELO / "metadata.json").write_text(json.dumps(a_python(metadata), indent=2, ensure_ascii=False))
    umbrales = {"version": version_umbrales, "version_modelo": version, "umbrales": resultado["umbrales"],
                "unidad": "probabilidad de fraude EN PRODUCCIÓN (ya calibrada)", "reglas_duras": ["lista_negra:solicitud_cumplimiento"],
                "creado_por": "notebook de entrenamiento (propuesta)", "creado_en": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "motivo": "mínimo costo en validación con revisiones ≤ capacidad de analistas y FP ≤ 15%",
                "autorizado_por": "PENDIENTE: gerencia de riesgo (DA5 / RO5)"}
    (RUTA_MODELO / "umbrales.json").write_text(json.dumps(a_python(umbrales), indent=2, ensure_ascii=False))
    return metadata

meta = exportar(red_nam, res_nam, version="2.0.0", descripcion="Red aditiva explicable (TensorFlow) para decisión aprobar/revisar/bloquear")
pd.Series(meta["metricas_prueba"]).to_frame("prueba")

### Simular lo que hará la API
La API no tiene TensorFlow: solo `onnxruntime`, el preprocesador y las tablas de perfil/dispositivo/lista negra. Cargamos los artefactos **desde disco** y decidimos una transacción nueva, como llegaría del core (sin ninguno de los campos que no existen al autorizar).

In [ ]:
prep_api = joblib.load(RUTA_MODELO / "preprocesador.joblib")
sesion_api = ort.InferenceSession(str(RUTA_MODELO / "modelo.onnx"), providers=["CPUExecutionProvider"])
meta_api = json.loads((RUTA_MODELO / "metadata.json").read_text())
umb_api = json.loads((RUTA_MODELO / "umbrales.json").read_text())

cal_api = Calibrador()
cal_api.a, cal_api.b = meta_api["calibracion"]["a"], meta_api["calibracion"]["b"]
cal_api.log_razon_prior = meta_api["calibracion"]["log_razon_prior"]

CTX_API = {"prep": prep_api, "usr": usr, "dsp": dsp, "lneg": lneg, "cal": cal_api, "umbrales": umb_api["umbrales"],
           "base_aportes": np.array(meta_api["aportes_base"]), "factores": meta_api["factores"],
           "version_modelo": meta_api["version"], "version_umbrales": umb_api["version"]}
scorer_onnx = lambda A: salidas_onnx(sesion_api, A)

# Una transacción de fraude y una legítima de la prueba
tid_fraude = te.loc[te["y"] == 1, "transaccion_id"].iloc[0]
tid_legit = te.loc[(te["y"] == 0) & te["etiquetada"], "transaccion_id"].iloc[0]
for etiqueta, tid in [("fraude confirmado", tid_fraude), ("legítima confirmada", tid_legit)]:
    res, aud = decidir(registro_json(tid), scorer_onnx, CTX_API, top=3)
    print(f"\n[{etiqueta}] {tid} → {res['decision'].upper()} (p = {res['probabilidad_fraude']:.2e})")
    for r in res["razones"]:
        print(f"   · {r['descripcion']}: valor {r.get('valor')} → {r['efecto']} ({r.get('aporte_logodds')})")

# Latencia de la decisión completa, una transacción a la vez (RNF1: p99 < 100 ms). No incluye red ni consulta a almacenes.
muestra = te["transaccion_id"].sample(300, random_state=SEMILLA).tolist()
lat = []
for tid in muestra:
    reg = registro_json(tid)
    t0 = time.perf_counter()
    decidir(reg, scorer_onnx, CTX_API)
    lat.append((time.perf_counter() - t0) * 1000)
print(f"\nLatencia por decisión (300 transacciones): p50 {np.percentile(lat, 50):.1f} ms · p99 {np.percentile(lat, 99):.1f} ms · máx {max(lat):.1f} ms")
print("Ojo: aquí el perfil y la lista negra están en memoria; en producción son consultas a un almacén de baja latencia (RT7).")

In [ ]:
import shutil
shutil.make_archive(str(CARPETA / "modelo_fraude"), "zip", RUTA_MODELO)
try:
    from google.colab import files
    files.download(str(CARPETA / "modelo_fraude.zip"))
except ImportError:
    print("No estás en Colab: el zip quedó en", CARPETA / "modelo_fraude.zip")

## 14 · Lo que este notebook NO resuelve (y dónde se resuelve)

| Pendiente | Por qué no se puede cerrar aquí | Dónde se cierra |
|---|---|---|
| Que la tasa de falsos positivos sobre bloqueos sea < 15% | La muestra tiene decenas de fraudes; un solo falso positivo pesa decenas por día (sección 10) | **Modo sombra**, 8 semanas, tráfico real (RO3) |
| Fraude no detectado ≤ el del motor actual | No tenemos las decisiones del motor sobre estas transacciones | Modo sombra: ambos deciden sobre el mismo tráfico (RF9) |
| Etiquetas: parte del 96% «no revisado» puede ser fraude | Nadie lo confirmó (R6) | Ciclo de etiquetado: RF6 + muestreo de transacciones aprobadas para revisión |
| Velocidad y dispositivo compartido en línea | No se pueden recalcular desde una muestra estratificada ni desde Redshift (RT7) | Almacén de baja latencia alimentado por el flujo de transacciones |
| Listas de sanciones internacionales (RF8) | No vienen en los datos | Regla dura con archivo diario (RT6) |
| Definición exacta de la regla dura de lista negra | Es una decisión normativa; con nuestro supuesto bloquearía a muchos clientes buenos (sección 10.0) | Cumplimiento + Riesgo, antes del modo sombra |
| Costo de revisión, efectividad del analista, tipo de cambio | Son supuestos de este notebook | Confirmar con Riesgo y Finanzas; se cambian en un solo lugar (celda de constantes) |
| El fraude se adapta (R5) | Un modelo entrenado hoy envejece | Alerta de desvío de la tasa de bloqueo (RF12) + reentrenamiento con la puerta de la sección 12 |
| Costo por millón de transacciones, capacidad 1.000.000/día | Se mide con carga, no en un notebook | Prueba de carga con el servicio (partes 2–4) |

### Para discutir antes de pasar a la API
1. La red aditiva no ve interacciones (p. ej. «monto alto **y** país distinto»). ¿Cuánto pierde frente al gradient boosting? ¿Compensa la explicación exacta? ¿Qué interacciones específicas agregarías como factores nuevos, manteniendo la explicación por factor?
2. De madrugada no hay zona «revisar». ¿Qué cambia si se **retiene** la transacción hasta las 08:00? ¿Quién paga ese costo (el cliente legítimo) y quién decide?
3. Si mañana el costo de un falso positivo sube de USD 6,9 a USD 14, ¿qué cambia: la red o los umbrales? ¿Hay que reconstruir la imagen?
4. Los umbrales se calcularon en validación con ~20 fraudes. ¿Qué tan estables son? ¿Cómo lo mostrarías a riesgo antes de que firme `umbrales.json`?
5. La regla dura bloquea más clientes buenos que todo el modelo. ¿Qué preguntas le harías a Cumplimiento para acotarla sin incumplir la norma?
6. ¿Por qué se excluye `usuarios_distintos` aunque casi no cueste rendimiento? ¿Qué habría que construir para poder usarla de forma legítima?

### Checklist antes de pasar a la parte 2
- [ ] Tengo `preprocesador.joblib`, `modelo.onnx`, `metadata.json` y `umbrales.json`
- [ ] El ONNX pasó la verificación (probabilidad **y** aportes) con `onnxruntime`
- [ ] Sé qué columnas se excluyeron por fuga y puedo explicar por qué
- [ ] Sé cuántas filas descartó la limpieza y por qué (R1–R9, `metadata.json → datos.limpieza`)
- [ ] Puedo explicar por qué los umbrales no van dentro del modelo y quién los autoriza
- [ ] Puedo explicar por qué este notebook **no** demuestra la meta de falsos positivos